In [ ]:
import os

# Skolni Azure OpenAI konfigurace pro Jupyter server.
# Tento server nemusi dovolovat vytvorit .env, proto se hodnoty nastavuji primo v prvni bunce.
os.environ["AZURE_API_URL"] = "https://krot0-mh4m0n6c-swedencentral.openai.azure.com/openai/deployments/gpt-5-mini-4/chat/completions?api-version=2025-01-01-preview"
os.environ["AZURE_API_KEY"] = "YOUR_SUPER_SECRET_API_KEY_BELONGS_HERE_:)"
os.environ["AZURE_OPENAI_MODEL"] = "gpt-5-mini"
os.environ["OFFLINE_GPT"] = "0"

print("OFFLINE_GPT:", os.environ["OFFLINE_GPT"])
print("API KEY EXISTS:", bool(os.environ.get("AZURE_API_KEY")))
print("URL:", os.environ["AZURE_API_URL"])


#### Co dělá první buňka

**Účel:** Připraví konfiguraci pro školní Azure OpenAI souhrny přímo v notebooku, bez souboru `.env`.

- Nastaví `AZURE_API_KEY`, `AZURE_API_URL`, `AZURE_OPENAI_MODEL` a `OFFLINE_GPT` do proměnných prostředí aktuálního kernelu.
- Používá školní Azure deployment `gpt-5-mini-4`, který odpovídá původní funkční konfiguraci.
- `OFFLINE_GPT="0"` aktivuje skutečné API volání; hodnota `1` vrací jen testovací placeholder bez síťového volání.

**Výstup:** Nastavené proměnné prostředí, které pozdější buňka `call_gpt` načte přes `os.getenv`.


# Analytická pipeline studentských anket - finální FIS školní Azure

Tato finální varianta zpracuje všechny předměty fakulty FIS, které splní minimální počet odpovědí, a exportuje jednu sadu CSV připravenou pro Power BI. GPT souhrny volá přes školní Azure OpenAI konfiguraci nastavenou v první buňce notebooku.


#### Co dělá následující buňka

**Účel:** Doplní knihovny, které nejsou součástí základní instalace Pythonu.

- `pyarrow` umožňuje knihovně pandas načíst zdrojový Parquet soubor.
- `reportlab` se používá v závěrečné buňce pro vytvoření PDF reportů s českým textem.

**Poznámka:** Příkazy `%pip` instalují balíčky do prostředí aktuálního Jupyter kernelu. Po první instalaci může být podle prostředí potřeba kernel restartovat.


In [ ]:
%pip install pyarrow
%pip install reportlab

#### Co dělá následující buňka

**Vstup:** Soubor `dataset_4_sentiment.parquet` určený proměnnou `DATA_PATH`.

**Postup:**
- Načte data pomocí pandas a ověří přítomnost povinných sloupců definovaných v `BASE_COLUMNS`.
- Sjednotí identifikaci vyučujících do polí `teacher_full_name` a `teacher_department`, aby další agregace pracovaly se stejnými názvy.
- Vytvoří `course_catalog`, tedy katalog unikátních předmětů s fakultou, garantujícím pracovištěm a počtem dostupných řádků.

**Výstupy:** Zdrojový DataFrame `df` a katalog `course_catalog`, ze kterého lze zvolit `selected_course_id`.


In [ ]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)

DATA_PATH = Path("dataset_4_sentiment.parquet")

# Keep the read narrow: the pipeline only loads columns used in validation, reporting and exports.
BASE_COLUMNS = [
    "obdobi",
    "kratka_programu_studenta",
    "nazev_programu_studenta",
    "fakulta_studenta",
    "nazev_oboru_studenta",
    "typ_studia_studenta",
    "forma_studia",
    "teacher_full_name",
    "teacher_department",
    "kod_predmetu_studenta",
    "nazev_predmetu_studenta",
    "fakulta_predmetu",
    "garantujici_pracoviste_predmetu",
    "otazka",
    "otazka_plny_text",
    "ciselne_hodnoceni",
    "textove_hodnoceni",
]

df = pd.read_parquet(DATA_PATH, columns=BASE_COLUMNS)
# normalize key filters to avoid trailing/leading whitespace mismatches
df["kratka_programu_studenta"] = df["kratka_programu_studenta"].str.strip()
df["kod_predmetu_studenta"] = df["kod_predmetu_studenta"].str.strip()
df["obdobi"] = df["obdobi"].str.strip()

# Course catalog is used for the FIS scope check and later Power BI course dimension.
course_catalog = (
    df.groupby([
        "kod_predmetu_studenta",
        "nazev_predmetu_studenta",
        "fakulta_predmetu",
        "garantujici_pracoviste_predmetu",
    ], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values(["rows", "kod_predmetu_studenta"], ascending=[False, True])
)
course_catalog.head(50)


## Přehled dostupných předmětů
Tabulka zobrazuje dostupné předměty, jejich fakultu a garantující pracoviště. Finální FIS varianta později automaticky vybere všechny řádky s `fakulta_predmetu = 40`.


### Metodická poznámka k výběru fakulty

Report je filtrovaný podle fakulty předmětu (`fakulta_predmetu = 40`, FIS). Studijní program studenta i garantující pracoviště zůstávají jako kontext v datech, ne jako hlavní filtr reportu.


## Výběr fakulty, historického okna a reportovaného semestru
`TARGET_FACULTY_CODE` a `TARGET_FACULTY_LABEL` určují fakultu FIS. Nastavení `selected_obdobi = None` zachová všechna období pro trendovou analýzu; `report_obdobi` určuje semestr detailních souhrnů.


#### Co dělá následující buňka

**Účel:** Soustředí hlavní uživatelská nastavení pipeline na jedno místo.

- `TARGET_FACULTY_CODE = 40` a `TARGET_FACULTY_LABEL = "FIS"` omezují zpracování na všechny předměty fakulty FIS.
- `selected_obdobi` určuje historické okno používané pro trendy. Hodnota `None` zachová všechna dostupná období.
- `report_obdobi` určuje semestr, pro který vzniknou detailní GPT souhrny a výsledné reporty.
- `REPORT_MIN_RESPONSE_COUNT` nastavuje minimální počet číselných odpovědí pro zahrnutí předmětu do GPT zpracování a exportů.
- Přepínače `EXPORT_TO_SQL` a `EXPORT_SQL_PREVIEW_FILES` řídí databázový export a lokální CSV/XLSX preview.

**Kontroly a výstup:** Normalizuje zadané hodnoty, ověří existenci fakulty FIS v `course_catalog` a vypíše počet nalezených předmětů i aktivní konfiguraci.


In [ ]:
# Faculty scope for this notebook variant. The source faculty code is exported as `faculty` in Power BI.
TARGET_FACULTY_CODE = 40
TARGET_FACULTY_LABEL = "FIS"
selected_course_id = f"faculty_{TARGET_FACULTY_LABEL}"  # scope label retained for shared report helpers

# Historical analysis window. None = all semesters available for the selected faculty.
# You can also use a list/tuple/set, e.g. ["ZS 2023/2024", "LS 2023/2024", "ZS 2024/2025"].
selected_obdobi = None

# Detailed report semester. None = latest semester from selected_obdobi/window.
report_obdobi = "LS 2024/2025"

# Optional SQL export for Azure SQL / Power BI. When False, the pipeline only creates Markdown/PDF as before.
EXPORT_TO_SQL = False
SQL_TABLE_PREFIX = "anketa_"
SQL_IF_EXISTS = "append"  # append is safest for report snapshots; use replace only in controlled test databases.

# Courses with fewer numeric answers are excluded from Markdown/PDF reports and Power BI export.
REPORT_MIN_RESPONSE_COUNT = 10

# Local preview of the exact aggregate tables that would be sent to SQL.
EXPORT_SQL_PREVIEW_FILES = True
SQL_PREVIEW_DIR = Path("powerbi_sql_preview") / TARGET_FACULTY_LABEL.lower()

TARGET_FACULTY_LABEL = TARGET_FACULTY_LABEL.strip()
if isinstance(selected_obdobi, str):
    selected_obdobi = selected_obdobi.strip()
elif selected_obdobi is not None:
    selected_obdobi = [str(x).strip() for x in selected_obdobi if str(x).strip()]

report_obdobi = report_obdobi.strip() if report_obdobi is not None else None

if not TARGET_FACULTY_LABEL:
    raise ValueError("Set TARGET_FACULTY_LABEL to a non-empty label.")

target_faculty_catalog = course_catalog[
    course_catalog["fakulta_predmetu"].eq(TARGET_FACULTY_CODE)
].copy()
if target_faculty_catalog.empty:
    raise ValueError(f"No courses for faculty code {TARGET_FACULTY_CODE!r}. Check course_catalog.")

print(f"Selected faculty: {TARGET_FACULTY_LABEL} (fakulta_predmetu={TARGET_FACULTY_CODE})")
print(f"Courses found before report-semester response threshold: {target_faculty_catalog['kod_predmetu_studenta'].nunique():,}")
display(target_faculty_catalog)
print(f"Historical window: {selected_obdobi or 'ALL available semesters'}")
print(f"Requested report semester: {report_obdobi or 'latest available'}")
print(f"SQL export enabled: {EXPORT_TO_SQL}")
print(f"SQL preview files enabled: {EXPORT_SQL_PREVIEW_FILES}")
print(f"Report minimum numeric responses for the course: {REPORT_MIN_RESPONSE_COUNT}")


## Lokální preprocessing, validace a split dat

Tato část běží lokálně na Jupyter serveru: filtruje předmět/období, čistí textové odpovědi, validuje číselná hodnocení a připravuje oddělené řezy pro další agregace. OpenAI API se zde nepoužívá.


#### Co dělá následující buňka

**Vstup:** Zdrojový DataFrame `df`, zvolená fakulta `TARGET_FACULTY_CODE` / `TARGET_FACULTY_LABEL` a historické okno `selected_obdobi`.

**Postup:**
- Definuje očekávané reportovací sloupce, platný rozsah hodnocení a seznam prázdných či zástupných hodnot.
- Funkce pro textová data normalizují odpovědi, odstraňují prázdné nebo bezvýznamné komentáře a evidují důvody vyřazení bez zveřejnění platných surových komentářů.
- Funkce pro číselná data převádějí hodnoty na čísla a odstraňují chybějící, nečíselná nebo mimoškálová hodnocení.
- `split_course_data` připravuje jeden předmět; `prepare_all_courses` používá stejné kontroly bez filtru na předmět.
- Vstupní data omezí na všechny předměty, jejichž `fakulta_predmetu` odpovídá hodnotě `TARGET_FACULTY_CODE`.

**Výstupy:** `prepared`, `text_prepared`, `num_prepared`, `filtered` a agregované reporty kvality dat. Tyto objekty jsou základem všech dalších analýz.


In [ ]:
from collections import Counter
import re
import string
import warnings

REPORT_COLUMNS = [
    "obdobi",
    "kratka_programu_studenta",
    "nazev_programu_studenta",
    "nazev_oboru_studenta",
    "fakulta_studenta",
    "typ_studia_studenta",
    "forma_studia",
    "kod_predmetu_studenta",
    "nazev_predmetu_studenta",
    "fakulta_predmetu",
    "garantujici_pracoviste_predmetu",
    "teacher_full_name",
    "teacher_department",
    "otazka",
    "otazka_plny_text",
    "ciselne_hodnoceni",
    "textove_hodnoceni",
]

TEXT_COLUMN = "textove_hodnoceni"
NUMERIC_COLUMN = "ciselne_hodnoceni"
EXPECTED_RATING_MIN = 1
EXPECTED_RATING_MAX = 4

# Explicit placeholder lists keep text cleaning reproducible and easy to audit.
NULL_TOKENS = {
    "", "null", "[null]", "none", "nan", "na", "n/a", "nezadano", "neuvedeno", "bez odpovedi",
}
PLACEHOLDER_TOKENS = {
    ".", "..", "...", ",", "-", "--", "---", "_", "__", "/", "\\", "?", "??", "???",
    "x", "xx", "xxx", "*", "**", "***", "0", "00",
}
VALID_SHORT_ANSWERS = {"ano", "ne", "nic", "dobre", "dobré", "super", "ok", "oki", "jo"}
FILLER_TOKENS = {"test", "asdf", "qwerty", "bla", "blabla", "aaaa", "bbbb", "cccc", "xxx", "xxxx"}
PUNCTUATION_CHARS = set(string.punctuation) | {"…", "–", "—", "„", "“", "”", "‚", "‘", "’", "·", "•"}


def normalize_text_value(value) -> str:
    """Return a stripped string representation; missing values become an empty string."""
    if pd.isna(value):
        return ""
    return str(value).replace("\u00a0", " ").strip()


def classify_text_response(value) -> str | None:
    """Return None for meaningful text, otherwise a reason why the response is invalid."""
    text = normalize_text_value(value)
    token = text.casefold()

    if text == "" or token in NULL_TOKENS:
        return "missing_or_null_placeholder"
    if token in VALID_SHORT_ANSWERS:
        return None
    if token in PLACEHOLDER_TOKENS:
        return "symbol_placeholder"
    if token in FILLER_TOKENS:
        return "known_filler"
    if len(set(token)) == 1 and len(token) >= 3:
        return "repeated_single_character"
    if all(ch.isspace() or ch in PUNCTUATION_CHARS for ch in text):
        return "punctuation_only"
    if re.fullmatch(r"[\d\s.,:/\\+-]+", text):
        return "numeric_or_date_like"
    if not re.search(r"[A-Za-zÁ-ž]", text):
        return "no_letters"
    # One-letter or two-letter fragments are usually not useful feedback, while explicit valid short answers are kept above.
    alpha_chars = re.findall(r"[A-Za-zÁ-ž]", text)
    if len(alpha_chars) <= 2 and token not in VALID_SHORT_ANSWERS:
        return "too_short_fragment"
    # Repeated non-word filler such as "haha", "jeje" can still be meaningful in rare cases; keep it unless it is already a known filler.
    return None


def is_meaningful_text_response(value) -> bool:
    """True only for responses that look like meaningful natural-language feedback."""
    return classify_text_response(value) is None


# Text validation removes empty placeholders and technical noise, but keeps meaningful short answers.
def clean_text_responses(df: pd.DataFrame, text_column: str = TEXT_COLUMN):
    """Clean text responses once and return the cleaned frame plus aggregate quality statistics."""
    original_count = len(df)
    work = df.copy()
    work[text_column] = work[text_column].map(normalize_text_value)
    work["_text_invalid_reason"] = work[text_column].map(classify_text_response)

    removed = work[work["_text_invalid_reason"].notna()].copy()
    cleaned = work[work["_text_invalid_reason"].isna()].copy().drop(columns=["_text_invalid_reason"])

    removed_values = [normalize_text_value(v).casefold() or "<empty>" for v in removed[text_column]]
    quality = {
        "original_text_responses": int(original_count),
        "removed_invalid_text_responses": int(len(removed)),
        "valid_text_responses": int(len(cleaned)),
        "percentage_removed": round((len(removed) / original_count * 100), 2) if original_count else 0.0,
        "removed_reasons": removed["_text_invalid_reason"].value_counts().to_dict(),
        # Only aggregate placeholder-like removed values. Do not log valid raw comments.
        "most_common_removed_placeholders": Counter(removed_values).most_common(10),
    }
    return cleaned, quality, removed[[text_column, "_text_invalid_reason"]]


# Numeric validation keeps only ratings on the expected 1-4 survey scale.
def validate_numeric_responses(
    df: pd.DataFrame,
    numeric_column: str = NUMERIC_COLUMN,
    expected_min: int = EXPECTED_RATING_MIN,
    expected_max: int = EXPECTED_RATING_MAX,
):
    """Coerce and validate numeric ratings for the expected survey scale."""
    original_count = len(df)
    work = df.copy()
    raw = work[numeric_column]
    numeric = pd.to_numeric(raw, errors="coerce")

    missing_mask = raw.isna() | raw.map(lambda v: normalize_text_value(v).casefold() in NULL_TOKENS)
    invalid_string_mask = numeric.isna() & ~missing_mask
    out_of_scale_mask = numeric.notna() & ~numeric.between(expected_min, expected_max)

    invalid_examples = (
        raw[invalid_string_mask | out_of_scale_mask]
        .map(normalize_text_value)
        .replace("", "<empty>")
        .value_counts()
        .head(10)
        .to_dict()
    )

    quality = {
        "original_numeric_responses": int(original_count),
        "missing_numeric_responses": int(missing_mask.sum()),
        "invalid_string_numeric_responses": int(invalid_string_mask.sum()),
        "out_of_scale_numeric_responses": int(out_of_scale_mask.sum()),
        "valid_numeric_responses": int((numeric.notna() & numeric.between(expected_min, expected_max)).sum()),
        "expected_scale": f"{expected_min}-{expected_max}",
        "most_common_invalid_numeric_values": invalid_examples,
    }

    cleaned = work[numeric.notna() & numeric.between(expected_min, expected_max)].copy()
    cleaned[numeric_column] = numeric[numeric.notna() & numeric.between(expected_min, expected_max)].astype(float)

    if quality["invalid_string_numeric_responses"] or quality["out_of_scale_numeric_responses"]:
        warnings.warn(
            "Some numeric survey responses were removed because they were not numeric or outside "
            f"the expected {expected_min}-{expected_max} scale. See numeric_quality_report.",
            RuntimeWarning,
        )
    return cleaned, quality


def print_quality_report(text_quality: dict, numeric_quality: dict):
    """Print aggregate data quality diagnostics without exposing raw student comments."""
    print("Text data quality:")
    print(f"- original text responses: {text_quality['original_text_responses']:,}")
    print(f"- removed invalid text responses: {text_quality['removed_invalid_text_responses']:,}")
    print(f"- valid text responses: {text_quality['valid_text_responses']:,}")
    print(f"- percentage removed: {text_quality['percentage_removed']:.2f}%")
    print(f"- removed reasons: {text_quality['removed_reasons']}")
    print(f"- common removed placeholders: {text_quality['most_common_removed_placeholders']}")
    print(f"- empty/null text cells in filtered data: {text_quality.get('empty_text_cells_in_filtered_data', 0):,}")
    print("Numeric data quality:")
    print(f"- original numeric candidates: {numeric_quality['original_numeric_responses']:,}")
    print(f"- valid numeric responses: {numeric_quality['valid_numeric_responses']:,}")
    print(f"- missing numeric responses: {numeric_quality['missing_numeric_responses']:,}")
    print(f"- invalid strings: {numeric_quality['invalid_string_numeric_responses']:,}")
    print(f"- out of expected scale ({numeric_quality['expected_scale']}): {numeric_quality['out_of_scale_numeric_responses']:,}")
    print(f"- common invalid numeric values: {numeric_quality['most_common_invalid_numeric_values']}")


def split_course_data(source_df: pd.DataFrame, course_id: str, obdobi=None):
    """Filter locally by course ID, normalize placeholders, clean text, validate numeric ratings, then split report slices."""
    course_id = course_id.strip()
    if isinstance(obdobi, str):
        obdobi = obdobi.strip()
    elif obdobi is not None:
        obdobi = [str(x).strip() for x in obdobi if str(x).strip()]
    if not course_id:
        raise ValueError("Set selected_course_id to one of the codes from course_catalog.")

    filtered = source_df[source_df["kod_predmetu_studenta"] == course_id].copy()
    if isinstance(obdobi, str):
        filtered = filtered[filtered["obdobi"] == obdobi]
    elif obdobi is not None:
        filtered = filtered[filtered["obdobi"].isin(obdobi)]

    if filtered.empty:
        raise ValueError(f"No data for course ID {course_id} and obdobi {obdobi or 'ALL'}.")

    text_tokens = filtered[TEXT_COLUMN].map(normalize_text_value).str.casefold()
    text_null = filtered[TEXT_COLUMN].isna() | text_tokens.isin(NULL_TOKENS)
    filtered.loc[text_null, TEXT_COLUMN] = pd.NA

    num_tokens = filtered[NUMERIC_COLUMN].map(normalize_text_value).str.casefold()
    num_null = filtered[NUMERIC_COLUMN].isna() | num_tokens.isin(NULL_TOKENS)
    filtered.loc[num_null, NUMERIC_COLUMN] = pd.NA

    empty_text_cells = filtered[TEXT_COLUMN].map(normalize_text_value).eq("").sum()
    text_candidates = filtered[filtered[NUMERIC_COLUMN].isna() & filtered[TEXT_COLUMN].notna()].copy()
    text_clean, text_quality_report, removed_text_values = clean_text_responses(text_candidates)
    text_quality_report["empty_text_cells_in_filtered_data"] = int(empty_text_cells)
    text_output = text_clean.drop(columns=[NUMERIC_COLUMN])

    numeric_candidates = filtered[filtered[TEXT_COLUMN].isna() & filtered[NUMERIC_COLUMN].notna()].copy()
    num_clean, numeric_quality_report = validate_numeric_responses(numeric_candidates)
    num_output = num_clean.drop(columns=[TEXT_COLUMN])

    prepared_parts = []
    if not text_clean.empty:
        prepared_parts.append(text_clean.drop(columns=[NUMERIC_COLUMN], errors="ignore"))
    if not num_clean.empty:
        prepared_parts.append(num_clean.drop(columns=[TEXT_COLUMN], errors="ignore"))
    if prepared_parts:
        prepared = pd.concat(prepared_parts, ignore_index=True, sort=False).reindex(columns=REPORT_COLUMNS).drop_duplicates()
    else:
        prepared = pd.DataFrame(columns=REPORT_COLUMNS)

    return prepared, text_output, num_output, filtered, text_quality_report, numeric_quality_report, removed_text_values




def prepare_all_courses(source_df, obdobi=None):
    """Stejne cisteni/validace jako split_course_data, ale BEZ filtru na jeden predmet.

    Vraci prepared / text_prepared / num_prepared pro VSECHNY predmety. Znovu pouziva
    clean_text_responses a validate_numeric_responses, takze vsechny guardy zustavaji.
    """
    if isinstance(obdobi, str):
        obdobi = obdobi.strip()
    elif obdobi is not None:
        obdobi = [str(x).strip() for x in obdobi if str(x).strip()]

    filtered = source_df.copy()
    if isinstance(obdobi, str):
        filtered = filtered[filtered["obdobi"] == obdobi]
    elif obdobi is not None:
        filtered = filtered[filtered["obdobi"].isin(obdobi)]

    if filtered.empty:
        raise ValueError(f"Zadna data pro obdobi {obdobi or 'ALL'}.")

    text_tokens = filtered[TEXT_COLUMN].map(normalize_text_value).str.casefold()
    text_null = filtered[TEXT_COLUMN].isna() | text_tokens.isin(NULL_TOKENS)
    filtered.loc[text_null, TEXT_COLUMN] = pd.NA

    num_tokens = filtered[NUMERIC_COLUMN].map(normalize_text_value).str.casefold()
    num_null = filtered[NUMERIC_COLUMN].isna() | num_tokens.isin(NULL_TOKENS)
    filtered.loc[num_null, NUMERIC_COLUMN] = pd.NA

    empty_text_cells = filtered[TEXT_COLUMN].map(normalize_text_value).eq("").sum()

    text_candidates = filtered[filtered[NUMERIC_COLUMN].isna() & filtered[TEXT_COLUMN].notna()].copy()
    text_clean, text_quality_report, removed_text_values = clean_text_responses(text_candidates)
    text_quality_report["empty_text_cells_in_filtered_data"] = int(empty_text_cells)
    text_output = text_clean.drop(columns=[NUMERIC_COLUMN])

    numeric_candidates = filtered[filtered[TEXT_COLUMN].isna() & filtered[NUMERIC_COLUMN].notna()].copy()
    num_clean, numeric_quality_report = validate_numeric_responses(numeric_candidates)
    num_output = num_clean.drop(columns=[TEXT_COLUMN])

    prepared_parts = []
    if not text_clean.empty:
        prepared_parts.append(text_clean.drop(columns=[NUMERIC_COLUMN], errors="ignore"))
    if not num_clean.empty:
        prepared_parts.append(num_clean.drop(columns=[TEXT_COLUMN], errors="ignore"))
    if prepared_parts:
        prepared = pd.concat(prepared_parts, ignore_index=True, sort=False).reindex(columns=REPORT_COLUMNS).drop_duplicates()
    else:
        prepared = pd.DataFrame(columns=REPORT_COLUMNS)

    return prepared, text_output, num_output, filtered, text_quality_report, numeric_quality_report, removed_text_values


# This is the main FIS filter: all later calculations use only courses with fakulta_predmetu = 40.
analysis_source_df = df[
    df["fakulta_predmetu"].eq(TARGET_FACULTY_CODE)
].copy()
print(f"Rezim zpracovani: vsechny predmety fakulty {TARGET_FACULTY_LABEL} (fakulta_predmetu={TARGET_FACULTY_CODE})")

(
    prepared,
    text_prepared,
    num_prepared,
    filtered,
    text_quality_report,
    numeric_quality_report,
    removed_text_values,
) = prepare_all_courses(analysis_source_df, selected_obdobi)

n_courses = prepared["kod_predmetu_studenta"].nunique()
print(f"Pripraveno predmetu: {n_courses:,}")
print(f"Radku celkem: {len(filtered):,} | text: {len(text_prepared):,} | numeric: {len(num_prepared):,}")
print_quality_report(text_quality_report, numeric_quality_report)
prepared.head()


### Validation: splits and dropped columns


#### Co dělá následující buňka

**Účel:** Provádí rychlé regresní kontroly výsledku preprocessingu před spuštěním dražších analýz a GPT volání.

- Ověří, že `text_prepared` neobsahuje číselný sloupec a `num_prepared` neobsahuje textový sloupec.
- Zkontroluje, že v textovém výřezu nezůstaly placeholdery nebo neplatné komentáře.
- Zkontroluje, že číselný výřez neobsahuje hodnoty `NULL` ani hodnocení mimo očekávanou škálu.

**Výstup:** Při úspěchu vypíše počty řádků. Při porušení podmínek se běh zastaví pomocí `assert`, aby se chybná data nedostala do reportů.


In [ ]:
# Ensure each slice only has relevant columns and the consolidated validators did their job.
assert NUMERIC_COLUMN not in text_prepared.columns, 'text_prepared should not include ciselne_hodnoceni.'
assert TEXT_COLUMN not in num_prepared.columns, 'num_prepared should not include textove_hodnoceni.'

if not text_prepared.empty:
    invalid_text_mask = ~text_prepared[TEXT_COLUMN].map(is_meaningful_text_response)
    assert invalid_text_mask.sum() == 0, 'Invalid or placeholder text rows slipped through.'

assert not num_prepared[NUMERIC_COLUMN].isna().any(), 'Numeric slice contains NULL ratings.'
assert num_prepared[NUMERIC_COLUMN].between(EXPECTED_RATING_MIN, EXPECTED_RATING_MAX).all(), 'Numeric slice contains ratings outside the expected scale.'

print(f"Filtered rows total: {len(filtered):,}")
print(f"Text-only cleaned rows: {len(text_prepared):,}")
print(f"Numeric cleaned rows: {len(num_prepared):,}")

### Quick checks
Row counts by semester and course to sanity-check the selected course slice.


#### Co dělá následující buňka

**Účel:** Rychlá kontrola časového pokrytí připravených dat.

Seskupí `prepared` podle sloupce `obdobi`, spočítá počet řádků v každém semestru a výsledky seřadí podle názvu období. Výstup pomáhá odhalit chybějící nebo nečekaně slabě zastoupené semestry ještě před trendovou analýzou.


In [ ]:
prepared["obdobi"].value_counts().sort_index()

#### Co dělá následující buňka

**Účel:** Rychlá kontrola složení analyzovaných dat podle studijních programů.

Seskupí `prepared` podle kódu a názvu programu, spočítá četnosti a zobrazí dvacet největších skupin. Tento přehled je diagnostický; program studenta se nepoužívá jako hlavní filtr reportu předmětu.


In [ ]:
prepared.groupby(["kratka_programu_studenta", "nazev_programu_studenta"]).size().sort_values(ascending=False).head(20)


### Optional: save prepared data
Uncomment one of the lines below if you want to store the filtered data for later report generation.

#### Co dělá následující buňka

**Účel:** Nabízí volitelné uložení mezivýsledku po preprocessingu.

- Parquet zachová datové typy a je vhodný pro rychlé další zpracování.
- CSV je snadno čitelné v běžných nástrojích, ale nemusí přesně zachovat všechny datové typy.

**Poznámka:** Příkazy jsou záměrně zakomentované, aby běžný běh nevytvářel další soubory. Pro použití je nutné odkomentovat požadovaný řádek.


In [ ]:
# prepared.to_parquet("prepared_report_data.parquet", index=False)
# prepared.to_csv("prepared_report_data.csv", index=False, encoding="utf-8")

## Report generation: local analytics + OpenAI inference

Local code computes multi-semester aggregates, trend deltas, anomaly flags and report structure. OpenAI API is used only inside `call_gpt` to turn prepared prompts into narrative summaries.


#### Co dělá následující buňka

**Účel:** Izoluje veškerou komunikaci se školním Azure OpenAI API do jediné funkce `call_gpt`, zatímco výpočty, filtry a agregace zůstávají lokální.

**Postup:**
- Načte URL API, API klíč, název modelu a přepínač `OFFLINE_GPT` z proměnných prostředí nastavených v první buňce notebooku.
- V offline režimu vrací testovací text a neodesílá žádná data přes síť.
- V online režimu sestaví požadavek pro Azure chat completions, odešle připravené zprávy a vrátí text odpovědi modelu.
- Pomocná funkce `_azure_error_payload` zpracuje chybovou odpověď API a přidá ji do srozumitelné výjimky.

**Výstup:** Funkce `call_gpt(messages)`, kterou používají následující buňky pro textová a číselná shrnutí.

In [ ]:
import os
import requests
import json
import re
import numpy as np

# Azure OpenAI is intentionally isolated to this inference wrapper. All data filtering,
# validation, aggregation, trend analysis, anomaly indicators and report assembly run locally.
AZURE_API_URL = os.getenv(
    "AZURE_API_URL",
    "https://krot0-mh4m0n6c-swedencentral.openai.azure.com/openai/deployments/gpt-5-mini-4/chat/completions?api-version=2025-01-01-preview",
)
AZURE_API_KEY = os.getenv("AZURE_API_KEY")  # set in the first notebook cell
MODEL_NAME = os.getenv("AZURE_OPENAI_MODEL", "gpt-5-mini")

# Set OFFLINE_GPT=1 to test the notebook without network / API credentials.
OFFLINE_GPT = os.getenv("OFFLINE_GPT", "").strip().lower() in {"1", "true", "yes"}


def _azure_error_payload(response):
    try:
        return response.json()
    except ValueError:
        return {"error": {"message": response.text[:500]}}


def call_gpt(messages, timeout: int = 90):
    """Call Azure OpenAI / Azure AI Foundry. This is the only network inference boundary."""
    if OFFLINE_GPT:
        return "Pozn.: (OFFLINE_GPT) K dispozici je omezeny pocet odpovedi, vystup je pouze testovaci placeholder."
    if not AZURE_API_KEY:
        raise RuntimeError("Run the first notebook cell to set AZURE_API_KEY before calling the Azure OpenAI model.")

    headers = {
        "Content-Type": "application/json",
        "api-key": AZURE_API_KEY,
    }
    payload = {
        "model": MODEL_NAME,
        "messages": messages,
        "temperature": 0,
    }
    try:
        resp = requests.post(AZURE_API_URL, headers=headers, json=payload, timeout=timeout)
        if resp.status_code >= 400:
            error_payload = _azure_error_payload(resp)
            raise RuntimeError(
                "Azure OpenAI LLM inference failed. Local preprocessing results are still available. "
                f"Model={MODEL_NAME}. Detail: {error_payload}"
            )
        data = resp.json()
        return data["choices"][0]["message"]["content"].strip()
    except requests.RequestException as exc:
        response = getattr(exc, "response", None)
        detail = f" Detail: {_azure_error_payload(response)}" if response is not None else ""
        raise RuntimeError("Azure OpenAI LLM inference failed. Local preprocessing results are still available." + detail) from exc


print(f"Azure OpenAI model: {MODEL_NAME}")

### Prepare multi-semester analytical datasets
Build current-semester report slices plus historical aggregates by selected course, teacher and semester.


#### Co dělá následující buňka

**Vstupy:** Vyčištěné datasety `prepared`, `text_prepared`, `num_prepared` a nastavení `report_obdobi`.

**Postup:**
- Seřadí semestry chronologicky a zvolí `selected_report_obdobi`; pokud není požadované období zadáno, použije nejnovější dostupný semestr.
- Vytvoří reportové výřezy pouze pro detailně reportovaný semestr a zachová historická data pro trendové porovnání.
- Sestaví `text_batches`, které seskupují vyčištěné komentáře podle předmětu.
- Vypočítá číselné statistiky podle otázek, předmětů, vyučujících a semestrů.
- Lokálně spočítá pokrytí textových odpovědí a výskyty předem definovaných negativních motivů bez exportu surových komentářů.

**Výstupy:** Reportové výřezy a agregační tabulky používané trendovou analýzou, GPT souhrny a finálními exporty.


In [ ]:
SEMESTER_TERM_ORDER = {"ZS": 1, "LS": 2}


def semester_sort_key(value: str):
    """Sort labels like 'ZS 2023/2024' and 'LS 2024/2025' chronologically."""
    text = normalize_text_value(value)
    match = re.search(r"(ZS|LS)\s*(\d{4})\s*/\s*(\d{4})", text, flags=re.IGNORECASE)
    if match:
        term = match.group(1).upper()
        start_year = int(match.group(2))
        end_year = int(match.group(3))
        return (start_year, end_year, SEMESTER_TERM_ORDER.get(term, 0), text)
    return (9999, 9999, 9, text)


def choose_report_semester(prepared_df: pd.DataFrame, requested_obdobi: str | None) -> str:
    available = sorted(prepared_df["obdobi"].dropna().unique(), key=semester_sort_key)
    if not available:
        raise ValueError("No semester labels available after preprocessing.")
    if requested_obdobi is None:
        return available[-1]
    if requested_obdobi not in available:
        raise ValueError(f"Requested report_obdobi={requested_obdobi!r} is not in selected data. Available: {available}")
    return requested_obdobi


# Split the selected historical window into one detailed report semester and historical context.
selected_report_obdobi = choose_report_semester(prepared, report_obdobi)
report_prepared = prepared[prepared["obdobi"] == selected_report_obdobi].copy()
text_report_prepared = text_prepared[text_prepared["obdobi"] == selected_report_obdobi].copy()
num_report_prepared = num_prepared[num_prepared["obdobi"] == selected_report_obdobi].copy()

print(f"Report semester used for detailed summaries: {selected_report_obdobi}")
print(f"Historical semesters in selected window: {sorted(prepared['obdobi'].dropna().unique(), key=semester_sort_key)}")


# GPT receives one compact text batch per course, built only from already-cleaned responses.
def build_text_batches(df: pd.DataFrame, include_obdobi: bool = False):
    """Build per-course text batches from the already-cleaned text slice."""
    text_df = df.copy()
    if text_df.empty:
        return []
    invalid_mask = ~text_df[TEXT_COLUMN].map(is_meaningful_text_response)
    if invalid_mask.any():
        warnings.warn(
            f"Dropping {int(invalid_mask.sum())} invalid text responses found after preprocessing.",
            RuntimeWarning,
        )
        text_df = text_df[~invalid_mask].copy()

    group_cols = ['kod_predmetu_studenta', 'nazev_predmetu_studenta']
    if include_obdobi:
        group_cols = ['obdobi', *group_cols]

    batches = []
    for keys, group in text_df.groupby(group_cols):
        if include_obdobi:
            obdobi, kod, nazev = keys
        else:
            kod, nazev = keys
            obdobi = selected_report_obdobi
        batches.append({
            'obdobi': obdobi,
            'kod': kod,
            'nazev': nazev,
            'responses': group[TEXT_COLUMN].tolist(),
        })
    return batches


# Numeric aggregations are calculated locally; the model only receives finished statistics.
def build_numeric_stats(df: pd.DataFrame, by_semester: bool = False):
    """Aggregate validated numeric answers by course, question and optionally semester."""
    base_cols = ['kod_predmetu_studenta', 'nazev_predmetu_studenta', 'otazka', 'otazka_plny_text']
    group_cols = ['obdobi', *base_cols] if by_semester else base_cols
    out_cols = [*group_cols, 'count', 'mean', 'median', 'std', 'min', 'max']
    num_df = df.copy()
    if num_df.empty:
        return pd.DataFrame(columns=out_cols)
    assert num_df[NUMERIC_COLUMN].between(EXPECTED_RATING_MIN, EXPECTED_RATING_MAX).all()
    return (
        num_df.groupby(group_cols)[NUMERIC_COLUMN]
        .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
        .reset_index()
    )


def build_teacher_numeric_stats(df: pd.DataFrame, by_semester: bool = False):
    """Aggregate validated numeric answers by course, teacher and optionally semester."""
    base_cols = ['kod_predmetu_studenta', 'nazev_predmetu_studenta', 'teacher_full_name']
    group_cols = ['obdobi', *base_cols] if by_semester else base_cols
    out_cols = [*group_cols, 'teacher_department', 'count', 'mean', 'median', 'std', 'min', 'max']
    num_df = df.copy()
    if num_df.empty:
        return pd.DataFrame(columns=out_cols)
    assert num_df[NUMERIC_COLUMN].between(EXPECTED_RATING_MIN, EXPECTED_RATING_MAX).all()
    teacher_missing = num_df['teacher_full_name'].isna() | num_df['teacher_full_name'].astype(str).str.strip().eq('')
    num_df['teacher_full_name'] = num_df['teacher_full_name'].fillna('').astype(str).str.strip()
    num_df.loc[teacher_missing, 'teacher_full_name'] = 'missing_teacher_name'
    num_df['teacher_department'] = num_df['teacher_department'].fillna('').astype(str).str.strip()

    rating_stats = (
        num_df.groupby(group_cols)[NUMERIC_COLUMN]
        .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
        .reset_index()
    )
    dept_stats = (
        num_df.groupby(group_cols)['teacher_department']
        .agg(lambda s: next((x for x in s if x), ''))
        .reset_index()
    )
    return rating_stats.merge(dept_stats, on=group_cols, how='left')[out_cols]


def build_course_semester_stats(df: pd.DataFrame):
    """Course-level historical series from raw validated numeric answers."""
    cols = ['obdobi', 'kod_predmetu_studenta', 'nazev_predmetu_studenta', 'count', 'mean', 'median', 'std', 'min', 'max']
    if df.empty:
        return pd.DataFrame(columns=cols)
    return (
        df.groupby(['obdobi', 'kod_predmetu_studenta', 'nazev_predmetu_studenta'])[NUMERIC_COLUMN]
        .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
        .reset_index()
    )


NEGATIVE_MOTIF_PATTERNS = {
    # Czech terms are preserved and English equivalents are added for English-taught courses/comments.
    'communication': r'komunik|neodpov|arogan|neochot|nejasn|communication|unclear instructions?|late feedback|no response|unresponsive',
    'organization': r'organiz|chaos|termin|zmate|pozd[eě]|nestih|organization|organisation|chaotic|unclear structure|poorly organized|poorly organised',
    'materials': r'material|podklad|slid|prezentac|zdroj|materials?|slides?|study materials?|missing resources?|resources?',
    'assessment': r'zkou[sš]|test|hodnocen|nespraved|zapoc|zápoč|exam|grading|evaluation|test|unclear grading|assessment',
    'workload': r'naroc|nároč|moc prace|přet[ií]žen|pret[ií]žen|workload|too demanding|time[- ]consuming|overload|too much work',
    'clarity': r'nepochop|nejasn|vysvetl|vysvětl|srozumit|unclear|confusing|hard to understand|explanation|explain',
}


def build_negative_motif_stats(text_df: pd.DataFrame) -> pd.DataFrame:
    """Privacy-conscious motif counts; no raw comments are exposed."""
    columns = ['obdobi', 'kod_predmetu_studenta', 'nazev_predmetu_studenta', 'motif', 'count']
    rows = []
    if text_df.empty:
        return pd.DataFrame(columns=columns)
    for (obdobi, kod, nazev), group in text_df.groupby(['obdobi', 'kod_predmetu_studenta', 'nazev_predmetu_studenta']):
        comments = group[TEXT_COLUMN].map(lambda x: normalize_text_value(x).casefold())
        for motif, pattern in NEGATIVE_MOTIF_PATTERNS.items():
            count = int(comments.str.contains(pattern, regex=True, na=False).sum())
            if count:
                rows.append({'obdobi': obdobi, 'kod_predmetu_studenta': kod, 'nazev_predmetu_studenta': nazev, 'motif': motif, 'count': count})
    return pd.DataFrame(rows, columns=columns)


def build_text_coverage_stats(text_df: pd.DataFrame, num_df: pd.DataFrame) -> pd.DataFrame:
    text_counts = (
        text_df.groupby(['obdobi', 'kod_predmetu_studenta', 'nazev_predmetu_studenta'])[TEXT_COLUMN]
        .count()
        .rename('valid_text_count')
        .reset_index()
    ) if not text_df.empty else pd.DataFrame(columns=['obdobi', 'kod_predmetu_studenta', 'nazev_predmetu_studenta', 'valid_text_count'])
    numeric_counts = (
        num_df.groupby(['obdobi', 'kod_predmetu_studenta', 'nazev_predmetu_studenta'])[NUMERIC_COLUMN]
        .count()
        .rename('numeric_answer_count')
        .reset_index()
    ) if not num_df.empty else pd.DataFrame(columns=['obdobi', 'kod_predmetu_studenta', 'nazev_predmetu_studenta', 'numeric_answer_count'])
    return numeric_counts.merge(text_counts, on=['obdobi', 'kod_predmetu_studenta', 'nazev_predmetu_studenta'], how='outer').fillna(0)


text_batches = build_text_batches(text_report_prepared)
numeric_stats = build_numeric_stats(num_report_prepared)
teacher_numeric_stats = build_teacher_numeric_stats(num_report_prepared)
semester_numeric_stats = build_numeric_stats(num_prepared, by_semester=True)
teacher_semester_numeric_stats = build_teacher_numeric_stats(num_prepared, by_semester=True)
course_semester_stats = build_course_semester_stats(num_prepared)
text_coverage_stats = build_text_coverage_stats(text_prepared, num_prepared)
negative_motif_stats = build_negative_motif_stats(text_prepared)

len(text_batches), numeric_stats.shape, semester_numeric_stats.shape, course_semester_stats.shape, teacher_semester_numeric_stats.shape


### Uspora GPT volani (jen predmety nad prahem)

#### Co dělá následující buňka

**Účel:** Zabrání zbytečným GPT voláním pro předměty, které by byly kvůli nízkému počtu odpovědí stejně vyřazeny z výsledného reportu.

- Funkce `_courses_above_threshold` spočítá počet číselných odpovědí za reportovaný semestr pro každý předmět.
- Zachová pouze předměty s alespoň `REPORT_MIN_RESPONSE_COUNT` odpověďmi.
- Stejným seznamem předmětů omezí číselné i textové reportové výřezy, aby všechny další souhrny pracovaly se stejnou množinou.
- Po filtrování znovu vytvoří `text_batches`, `numeric_stats` a `teacher_numeric_stats`.

**Výstup:** Menší a konzistentní sada vstupů pro GPT a reporty. V testovacím režimu se typicky jedná pouze o zvolený předmět.


In [ ]:
# Uspora GPT: model poustime jen pro predmety, ktere stejne projdou REPORT_MIN_RESPONSE_COUNT.
# Orezeme vstupy do GPT smycek (bunky 24/26) a do batchu. Je to presne mnozina pro Batch API.

def _courses_above_threshold(num_report_df, min_count):
    counts = (num_report_df.groupby(["kod_predmetu_studenta", "nazev_predmetu_studenta"])[NUMERIC_COLUMN]
              .count().rename("n").reset_index())
    return counts[counts["n"] >= min_count][["kod_predmetu_studenta", "nazev_predmetu_studenta"]]

_keep_keys = _courses_above_threshold(num_report_prepared, REPORT_MIN_RESPONSE_COUNT)

num_report_prepared = num_report_prepared.merge(_keep_keys, on=["kod_predmetu_studenta", "nazev_predmetu_studenta"], how="inner")
text_report_prepared = text_report_prepared.merge(_keep_keys, on=["kod_predmetu_studenta", "nazev_predmetu_studenta"], how="inner")
text_batches = build_text_batches(text_report_prepared)
numeric_stats = build_numeric_stats(num_report_prepared)
teacher_numeric_stats = build_teacher_numeric_stats(num_report_prepared)
print(f"GPT pobezi jen pro predmety s >= {REPORT_MIN_RESPONSE_COUNT} odpovedmi: {len(_keep_keys):,}")


### Trend and anomaly indicators

Simple explainable local indicators for management reporting. They separate teaching-quality risk signals from data reliability warnings and avoid black-box scoring.


#### Co dělá následující buňka

**Účel:** Vypočítá deterministické trendy, rizika a omezení dat před tím, než se výsledky použijí v textových komentářích.

**Hlavní pravidla:** Prahové hodnoty v úvodu buňky určují například nízké skóre, výrazný mezisemestrální pokles, odchylku od historického průměru, vysokou variabilitu nebo minimální velikost vzorku.

**Postup:**
- `analyze_course_trends` porovnává aktuální předmět s předchozím semestrem, dlouhodobou historií a klouzavým průměrem.
- `analyze_teacher_trends` provádí obdobné výpočty pro jednotlivé vyučující a porovnává je také s výsledkem předmětu.
- `detect_course_anomalies` kombinuje číselné trendy, rozdíly mezi vyučujícími, pokrytí texty a opakované negativní motivy.
- Každý závěr je doplněn technickými příznaky rizik a hodnocením spolehlivosti, aby bylo možné oddělit skutečný signál od slabých dat.

**Výstupy:** `trend_summary_df`, `teacher_trend_df` a `anomaly_flags_df`, které se později převádějí do čitelných českých vět.


In [ ]:
# Thresholds below define local risk flags; they are not inferred by the LLM.
LOW_SCORE_THRESHOLD = 3.0
HIGH_STD_THRESHOLD = 1.15
TEACHER_GAP_THRESHOLD = 1.0
MIN_COUNT_FOR_ANOMALY = 10
MIN_TEXT_RESPONSES_FOR_COVERAGE = 1
SUDDEN_DROP_THRESHOLD = -0.40
BASELINE_DROP_THRESHOLD = -0.40
TEACHER_DROP_THRESHOLD = -0.40
TEACHER_BELOW_COURSE_THRESHOLD = -0.40
PERFECT_LOW_N_THRESHOLD = 8
STABLE_DELTA_THRESHOLD = 0.20
TREND_DELTA_THRESHOLD = 0.40
VOLATILITY_THRESHOLD = 0.35


def safe_float(value):
    return float(value) if pd.notna(value) else np.nan


def relative_change(delta, reference):
    if pd.isna(delta) or pd.isna(reference) or reference == 0:
        return np.nan
    return float(delta / abs(reference) * 100)


def has_significant_drop(delta, threshold: float = SUDDEN_DROP_THRESHOLD) -> bool:
    return pd.notna(delta) and float(delta) <= threshold


def has_significant_change(delta, threshold: float = abs(SUDDEN_DROP_THRESHOLD)) -> bool:
    return pd.notna(delta) and abs(float(delta)) >= threshold


def classify_trend_direction(
    current_count: int,
    history: pd.DataFrame,
    delta_vs_previous,
    delta_vs_baseline,
    series_std,
) -> str:
    if current_count < MIN_COUNT_FOR_ANOMALY:
        return 'insufficient_sample_size'
    if history.empty:
        return 'insufficient_history'
    if pd.notna(series_std) and series_std >= VOLATILITY_THRESHOLD:
        return 'volatile'
    if pd.notna(delta_vs_previous) and delta_vs_previous <= -TREND_DELTA_THRESHOLD:
        return 'deteriorating'
    if pd.notna(delta_vs_baseline) and delta_vs_baseline <= -TREND_DELTA_THRESHOLD:
        return 'deteriorating'
    if pd.notna(delta_vs_previous) and delta_vs_previous >= TREND_DELTA_THRESHOLD:
        return 'improving'
    if pd.notna(delta_vs_baseline) and delta_vs_baseline >= TREND_DELTA_THRESHOLD:
        return 'improving'
    if pd.notna(delta_vs_previous) and abs(delta_vs_previous) <= STABLE_DELTA_THRESHOLD:
        return 'stable'
    return 'insufficient_history'


def classify_trend_reliability(
    current_count: int,
    previous_count: int,
    history_semesters: int,
    missing_text_feedback: bool = False,
    current_std=np.nan,
    teacher_level_available: bool = True,
) -> str:
    if current_count < 10 or history_semesters < 2:
        return 'LOW'
    if previous_count and previous_count < 10:
        return 'LOW'
    penalty = 0
    if missing_text_feedback:
        penalty += 1
    if pd.notna(current_std) and current_std >= HIGH_STD_THRESHOLD:
        penalty += 1
    if not teacher_level_available:
        penalty += 1
    if history_semesters >= 3 and current_count >= 30 and previous_count >= 20 and penalty == 0:
        return 'HIGH'
    if history_semesters >= 3 and current_count >= 15 and previous_count >= 10 and penalty <= 1:
        return 'MEDIUM'
    return 'LOW'


def trend_interpretation(row) -> str:
    direction = row.get('trend_direction', 'insufficient_history')
    reliability = row.get('trend_reliability', 'LOW')
    current_count = int(row.get('current_count', 0) or 0)
    history_semesters = int(row.get('historical_semesters_available', 0) or 0)
    if direction == 'insufficient_sample_size':
        return f'Trend nelze hodnotit robustne: aktualni pocet odpovedi je n={current_count}, spolehlivost {reliability}.'
    if direction == 'insufficient_history':
        return f'Trend nelze spolehlive vyhodnotit: dostupnych predchozich obdobi je {history_semesters}, spolehlivost {reliability}.'
    if direction == 'volatile':
        return f'Dostupna data naznacuji kolisani mezi semestry; interpretovat opatrne, spolehlivost {reliability}.'
    if direction == 'deteriorating':
        return f'Dostupna data mohou naznacovat zhorseni; kvuli spolehlivosti {reliability} nejde o definitivni zaver.'
    if direction == 'improving':
        return f'Dostupna data mohou naznacovat zlepseni; zaver je nutne cist se spolehlivosti {reliability}.'
    if direction == 'stable':
        return f'Dostupna data neukazuji vyraznou zmenu; nejde o dukaz stabilni kvality, spolehlivost {reliability}.'
    return f'Trendova interpretace je omezena; spolehlivost {reliability}.'


def trend_flags_from_series(history: pd.DataFrame, current, delta_vs_previous, delta_vs_baseline, direction: str, reliability: str) -> tuple[list[str], list[str]]:
    """Separate main risk flags from reliability warnings using production thresholds."""
    risks = []
    warnings = []
    current_count = int(current['count'])
    current_mean = safe_float(current['mean'])

    if direction == 'insufficient_history':
        warnings.append('insufficient_history')
    if direction == 'insufficient_sample_size' or current_count < MIN_COUNT_FOR_ANOMALY:
        warnings.append(f'low_sample_trend_warning n={current_count}')
    if reliability == 'LOW':
        warnings.append('low_trend_reliability')

    # Main trend risks require n>=10 and a practically meaningful change. A large drop is shown
    # even when the current score is still above 3.0 (for example 3.8 -> 3.2).
    if current_count >= MIN_COUNT_FOR_ANOMALY:
        if has_significant_drop(delta_vs_previous):
            risks.append(f'sudden_drop_vs_previous_semester delta={delta_vs_previous:.2f}')
        if pd.notna(current_mean) and current_mean <= LOW_SCORE_THRESHOLD and has_significant_drop(delta_vs_baseline, BASELINE_DROP_THRESHOLD):
            risks.append(f'below_historical_baseline delta={delta_vs_baseline:.2f}')
        if direction == 'volatile':
            risks.append('volatile_scores')
        if len(history) >= 2:
            hist_means = history['mean'].tolist()
            if hist_means[-1] < hist_means[-2] and has_significant_change(delta_vs_previous):
                if delta_vs_previous > 0:
                    risks.append('improving_after_previous_drop')
                elif delta_vs_previous < 0:
                    risks.append('persistent_deterioration')
    return risks, warnings

def analyze_course_trends(course_semester_df: pd.DataFrame, text_coverage_df: pd.DataFrame, report_semester: str) -> pd.DataFrame:
    """Course-level historical trend metrics for the selected report semester."""
    columns = [
        'kod_predmetu_studenta', 'nazev_predmetu_studenta', 'report_obdobi',
        'current_count', 'current_mean', 'current_std',
        'previous_obdobi', 'previous_count', 'previous_mean', 'delta_vs_previous', 'relative_change_vs_previous_pct',
        'historical_average', 'historical_count', 'historical_semesters_available', 'delta_vs_historical_average', 'relative_change_vs_historical_pct',
        'rolling_3_semester_average', 'delta_vs_rolling_3_semester_average',
        'best_historical_semester', 'best_historical_mean', 'worst_historical_semester', 'worst_historical_mean',
        'trend_direction', 'trend_reliability', 'trend_risk_flags', 'trend_reliability_warnings', 'trend_interpretation', 'trend_note'
    ]
    if course_semester_df.empty:
        return pd.DataFrame(columns=columns)

    rows = []
    for (kod, nazev), group in course_semester_df.groupby(['kod_predmetu_studenta', 'nazev_predmetu_studenta']):
        group = group.sort_values('obdobi', key=lambda s: s.map(semester_sort_key)).reset_index(drop=True)
        if report_semester not in set(group['obdobi']):
            continue
        current_idx = int(group.index[group['obdobi'] == report_semester][-1])
        current = group.iloc[current_idx]
        history = group.iloc[:current_idx].copy()
        previous = history.iloc[-1] if not history.empty else None
        rolling_history = history.tail(3)

        historical_count = int(history['count'].sum()) if not history.empty else 0
        historical_average = np.average(history['mean'], weights=history['count']) if historical_count else np.nan
        rolling_average = np.average(rolling_history['mean'], weights=rolling_history['count']) if len(rolling_history) >= 3 else np.nan
        previous_count = int(previous['count']) if previous is not None else 0
        previous_mean = safe_float(previous['mean']) if previous is not None else np.nan
        delta_vs_previous = safe_float(current['mean']) - previous_mean if previous is not None else np.nan
        delta_vs_historical = safe_float(current['mean']) - historical_average if historical_count else np.nan
        delta_vs_rolling = safe_float(current['mean']) - rolling_average if pd.notna(rolling_average) else np.nan
        series_std = group['mean'].std() if len(group) >= 2 else np.nan

        best = group.loc[group['mean'].idxmax()] if not group.empty else None
        worst = group.loc[group['mean'].idxmin()] if not group.empty else None

        coverage = text_coverage_df[
            (text_coverage_df['obdobi'] == report_semester)
            & (text_coverage_df['kod_predmetu_studenta'] == kod)
            & (text_coverage_df['nazev_predmetu_studenta'] == nazev)
        ]
        missing_text_feedback = coverage.empty or int(coverage['valid_text_count'].sum()) == 0

        direction = classify_trend_direction(
            int(current['count']),
            history,
            delta_vs_previous,
            delta_vs_historical,
            series_std,
        )
        reliability = classify_trend_reliability(
            int(current['count']),
            previous_count,
            len(history),
            missing_text_feedback=missing_text_feedback,
            current_std=current['std'],
            teacher_level_available=True,
        )
        risk_flags, reliability_warnings = trend_flags_from_series(history, current, delta_vs_previous, delta_vs_historical, direction, reliability)

        row = {
            'kod_predmetu_studenta': kod,
            'nazev_predmetu_studenta': nazev,
            'report_obdobi': current['obdobi'],
            'current_count': int(current['count']),
            'current_mean': safe_float(current['mean']),
            'current_std': safe_float(current['std']),
            'previous_obdobi': previous['obdobi'] if previous is not None else '',
            'previous_count': previous_count,
            'previous_mean': previous_mean,
            'delta_vs_previous': safe_float(delta_vs_previous),
            'relative_change_vs_previous_pct': safe_float(relative_change(delta_vs_previous, previous_mean)),
            'historical_average': safe_float(historical_average),
            'historical_count': historical_count,
            'historical_semesters_available': int(len(history)),
            'delta_vs_historical_average': safe_float(delta_vs_historical),
            'relative_change_vs_historical_pct': safe_float(relative_change(delta_vs_historical, historical_average)),
            'rolling_3_semester_average': safe_float(rolling_average),
            'delta_vs_rolling_3_semester_average': safe_float(delta_vs_rolling),
            'best_historical_semester': best['obdobi'] if best is not None else '',
            'best_historical_mean': safe_float(best['mean']) if best is not None else np.nan,
            'worst_historical_semester': worst['obdobi'] if worst is not None else '',
            'worst_historical_mean': safe_float(worst['mean']) if worst is not None else np.nan,
            'trend_direction': direction,
            'trend_reliability': reliability,
            'trend_risk_flags': '; '.join(risk_flags),
            'trend_reliability_warnings': '; '.join(reliability_warnings),
        }
        row['trend_interpretation'] = trend_interpretation(row)
        row['trend_note'] = row['trend_interpretation']
        rows.append(row)
    return pd.DataFrame(rows, columns=columns)


def analyze_teacher_trends(teacher_semester_df: pd.DataFrame, course_trend_df: pd.DataFrame, report_semester: str) -> pd.DataFrame:
    """Teacher-level trend metrics where teacher identifiers are present in source data."""
    columns = [
        'kod_predmetu_studenta', 'nazev_predmetu_studenta', 'teacher_full_name', 'teacher_department', 'report_obdobi',
        'teacher_current_count', 'teacher_current_mean', 'teacher_previous_obdobi', 'teacher_previous_count', 'teacher_previous_mean',
        'teacher_delta_vs_previous', 'teacher_relative_change_vs_previous_pct',
        'teacher_historical_average', 'teacher_historical_count', 'teacher_semesters_available', 'teacher_delta_vs_historical_average',
        'teacher_trend_direction', 'teacher_trend_reliability', 'teacher_trend_flags', 'teacher_reliability_warnings', 'teacher_trend_interpretation'
    ]
    if teacher_semester_df.empty:
        return pd.DataFrame(columns=columns)

    rows = []
    for (kod, nazev, teacher), group in teacher_semester_df.groupby(['kod_predmetu_studenta', 'nazev_predmetu_studenta', 'teacher_full_name']):
        group = group.sort_values('obdobi', key=lambda s: s.map(semester_sort_key)).reset_index(drop=True)
        if report_semester not in set(group['obdobi']):
            continue
        current_idx = int(group.index[group['obdobi'] == report_semester][-1])
        current = group.iloc[current_idx]
        history = group.iloc[:current_idx].copy()
        previous = history.iloc[-1] if not history.empty else None
        historical_count = int(history['count'].sum()) if not history.empty else 0
        historical_average = np.average(history['mean'], weights=history['count']) if historical_count else np.nan
        previous_count = int(previous['count']) if previous is not None else 0
        previous_mean = safe_float(previous['mean']) if previous is not None else np.nan
        delta_vs_previous = safe_float(current['mean']) - previous_mean if previous is not None else np.nan
        delta_vs_historical = safe_float(current['mean']) - historical_average if historical_count else np.nan
        series_std = group['mean'].std() if len(group) >= 2 else np.nan

        direction = classify_trend_direction(int(current['count']), history, delta_vs_previous, delta_vs_historical, series_std)
        reliability = classify_trend_reliability(
            int(current['count']),
            previous_count,
            len(history),
            missing_text_feedback=False,
            current_std=current['std'],
            teacher_level_available=teacher != 'missing_teacher_name',
        )
        risk_flags, reliability_warnings = trend_flags_from_series(history, current, delta_vs_previous, delta_vs_historical, direction, reliability)
        if teacher == 'missing_teacher_name':
            reliability_warnings.append('missing_teacher_name')

        course_row = course_trend_df[
            (course_trend_df['kod_predmetu_studenta'] == kod)
            & (course_trend_df['nazev_predmetu_studenta'] == nazev)
        ]
        if int(current['count']) >= MIN_COUNT_FOR_ANOMALY and not course_row.empty and pd.notna(course_row.iloc[0]['current_mean']):
            diff_vs_course = safe_float(current['mean']) - safe_float(course_row.iloc[0]['current_mean'])
            if diff_vs_course <= TEACHER_BELOW_COURSE_THRESHOLD:
                risk_flags.append(f'teacher_below_course_average delta={diff_vs_course:.2f}')
        if int(current['count']) >= MIN_COUNT_FOR_ANOMALY and has_significant_drop(delta_vs_previous, TEACHER_DROP_THRESHOLD):
            risk_flags.append(f'teacher_score_drop delta={delta_vs_previous:.2f}')

        row = {
            'kod_predmetu_studenta': kod,
            'nazev_predmetu_studenta': nazev,
            'teacher_full_name': teacher,
            'teacher_department': current.get('teacher_department', ''),
            'report_obdobi': current['obdobi'],
            'teacher_current_count': int(current['count']),
            'teacher_current_mean': safe_float(current['mean']),
            'teacher_previous_obdobi': previous['obdobi'] if previous is not None else '',
            'teacher_previous_count': previous_count,
            'teacher_previous_mean': previous_mean,
            'teacher_delta_vs_previous': safe_float(delta_vs_previous),
            'teacher_relative_change_vs_previous_pct': safe_float(relative_change(delta_vs_previous, previous_mean)),
            'teacher_historical_average': safe_float(historical_average),
            'teacher_historical_count': historical_count,
            'teacher_semesters_available': int(len(history)),
            'teacher_delta_vs_historical_average': safe_float(delta_vs_historical),
            'teacher_trend_direction': direction,
            'teacher_trend_reliability': reliability,
            'teacher_trend_flags': '; '.join(risk_flags),
            'teacher_reliability_warnings': '; '.join(reliability_warnings),
        }
        if len(history) == 0:
            row['teacher_trend_interpretation'] = 'Trend nelze spolehlive vyhodnotit, protoze pro vyucujiciho je dostupne pouze jedno obdobi.'
        else:
            row['teacher_trend_interpretation'] = trend_interpretation({
                'trend_direction': direction,
                'trend_reliability': reliability,
                'current_count': int(current['count']),
                'historical_semesters_available': int(len(history)),
            })
        rows.append(row)
    return pd.DataFrame(rows, columns=columns)


def detect_course_anomalies(
    numeric_stats_df: pd.DataFrame,
    teacher_stats_df: pd.DataFrame,
    trend_df: pd.DataFrame,
    teacher_trend_df: pd.DataFrame,
    text_coverage_df: pd.DataFrame,
    motif_df: pd.DataFrame,
    report_semester: str,
) -> pd.DataFrame:
    """Create explainable course-level risk and reliability indicators."""
    rows = []
    course_keys = set()
    if not numeric_stats_df.empty:
        course_keys.update(zip(numeric_stats_df['kod_predmetu_studenta'], numeric_stats_df['nazev_predmetu_studenta']))
    if not trend_df.empty:
        course_keys.update(zip(trend_df['kod_predmetu_studenta'], trend_df['nazev_predmetu_studenta']))
    if not text_coverage_df.empty:
        report_coverage = text_coverage_df[text_coverage_df['obdobi'] == report_semester].copy()
        report_coverage = report_coverage[(report_coverage['numeric_answer_count'] > 0) | (report_coverage['valid_text_count'] > 0)]
        course_keys.update(zip(report_coverage['kod_predmetu_studenta'], report_coverage['nazev_predmetu_studenta']))

    for kod, nazev in sorted(course_keys, key=lambda x: (str(x[1]), str(x[0]))):
        risk_flags = []
        reliability_warnings = []

        group = numeric_stats_df[
            (numeric_stats_df['kod_predmetu_studenta'] == kod)
            & (numeric_stats_df['nazev_predmetu_studenta'] == nazev)
        ]
        total = int(group['count'].sum()) if not group.empty else 0
        weighted_mean = np.average(group['mean'], weights=group['count']) if total else np.nan
        max_std = group['std'].dropna().max() if not group.empty and group['std'].notna().any() else np.nan
        min_score = group['min'].min() if not group.empty else np.nan

        if total < MIN_COUNT_FOR_ANOMALY:
            reliability_warnings.append(f'low_sample_size n={total}')
        if total and pd.notna(weighted_mean) and weighted_mean == EXPECTED_RATING_MAX and min_score == EXPECTED_RATING_MAX and total <= PERFECT_LOW_N_THRESHOLD:
            reliability_warnings.append(f'suspiciously_perfect_low_n n={total}')
        if total >= MIN_COUNT_FOR_ANOMALY and pd.notna(weighted_mean) and weighted_mean <= LOW_SCORE_THRESHOLD:
            risk_flags.append(f'low_average_score mean={weighted_mean:.2f}')
        if total >= MIN_COUNT_FOR_ANOMALY and pd.notna(max_std) and max_std >= HIGH_STD_THRESHOLD:
            risk_flags.append(f'high_polarization std={max_std:.2f}')

        teacher_group = teacher_stats_df[
            (teacher_stats_df['kod_predmetu_studenta'] == kod)
            & (teacher_stats_df['nazev_predmetu_studenta'] == nazev)
        ]
        enough_teacher_data = teacher_group[teacher_group['count'] >= MIN_COUNT_FOR_ANOMALY]
        if len(enough_teacher_data) >= 2:
            teacher_gap = enough_teacher_data['mean'].max() - enough_teacher_data['mean'].min()
            if teacher_gap >= TEACHER_GAP_THRESHOLD:
                risk_flags.append(f'large_teacher_difference gap={teacher_gap:.2f}')
        if not teacher_group.empty and (teacher_group['teacher_full_name'] == 'missing_teacher_name').any():
            reliability_warnings.append('missing_teacher_name')

        trend_row = trend_df[
            (trend_df['kod_predmetu_studenta'] == kod)
            & (trend_df['nazev_predmetu_studenta'] == nazev)
        ]
        trend_note = ''
        if not trend_row.empty:
            tr = trend_row.iloc[0]
            trend_note = tr['trend_note']
            for flag in str(tr.get('trend_risk_flags', '')).split(';'):
                flag = flag.strip()
                if flag and int(tr.get('current_count', 0) or 0) >= MIN_COUNT_FOR_ANOMALY:
                    risk_flags.append(flag)
            for flag in str(tr.get('trend_reliability_warnings', '')).split(';'):
                flag = flag.strip()
                if flag:
                    reliability_warnings.append(flag)

        teacher_trend_group = teacher_trend_df[
            (teacher_trend_df['kod_predmetu_studenta'] == kod)
            & (teacher_trend_df['nazev_predmetu_studenta'] == nazev)
        ]
        if not teacher_trend_group.empty:
            for row in teacher_trend_group.itertuples():
                for flag in str(row.teacher_trend_flags).split(';'):
                    flag = flag.strip()
                    if flag and int(row.teacher_current_count) >= MIN_COUNT_FOR_ANOMALY:
                        risk_flags.append(f'{row.teacher_full_name}: {flag}')
                if 'missing_teacher_name' in str(row.teacher_reliability_warnings):
                    reliability_warnings.append('missing_teacher_name')

        coverage = text_coverage_df[
            (text_coverage_df['obdobi'] == report_semester)
            & (text_coverage_df['kod_predmetu_studenta'] == kod)
            & (text_coverage_df['nazev_predmetu_studenta'] == nazev)
        ]
        valid_text_count = int(coverage['valid_text_count'].sum()) if not coverage.empty else 0
        numeric_answer_count = int(coverage['numeric_answer_count'].sum()) if not coverage.empty else total
        if numeric_answer_count > 0 and valid_text_count < MIN_TEXT_RESPONSES_FOR_COVERAGE:
            reliability_warnings.append('missing_text_feedback')

        motif_group = motif_df[
            (motif_df['obdobi'] == report_semester)
            & (motif_df['kod_predmetu_studenta'] == kod)
            & (motif_df['nazev_predmetu_studenta'] == nazev)
        ]
        motif_summary = ''
        if not motif_group.empty:
            motif_summary = ', '.join(f"{row.motif}:{int(row.count)}" for row in motif_group.itertuples())
            if total >= MIN_COUNT_FOR_ANOMALY and motif_group['count'].sum() >= 2:
                risk_flags.append(f'repeated_negative_motifs {motif_summary}')

        risk_flags = sorted(set(risk_flags))
        reliability_warnings = sorted(set(reliability_warnings))
        rows.append({
            'kod_predmetu_studenta': kod,
            'nazev_predmetu_studenta': nazev,
            'potential_risks': '; '.join(risk_flags),
            'data_reliability_warnings': '; '.join(reliability_warnings),
            'anomaly_flags': '; '.join([*risk_flags, *reliability_warnings]),
            'trend_note': trend_note,
            'valid_text_count': valid_text_count,
            'numeric_answer_count': numeric_answer_count,
            'negative_motif_summary': motif_summary,
        })
    return pd.DataFrame(rows)


# Build deterministic trend/risk tables before any narrative report text is assembled.
trend_summary_df = analyze_course_trends(course_semester_stats, text_coverage_stats, selected_report_obdobi)
teacher_trend_df = analyze_teacher_trends(teacher_semester_numeric_stats, trend_summary_df, selected_report_obdobi)
anomaly_flags_df = detect_course_anomalies(
    numeric_stats,
    teacher_numeric_stats,
    trend_summary_df,
    teacher_trend_df,
    text_coverage_stats,
    negative_motif_stats,
    selected_report_obdobi,
)
anomaly_flags_df[anomaly_flags_df['anomaly_flags'].ne('')].head(20)


### GPT summaries of text answers (report semester)
Only the selected report semester comments are sent to the model. Historical text analytics remain local and aggregated.


#### Co dělá následující buňka

**Vstup:** `text_batches`, tedy vyčištěné a anonymně seskupené komentáře za `selected_report_obdobi` pro předměty nad minimálním prahem odpovědí.

**Postup:**
- Funkce `summarize_text_course` očísluje komentáře a připraví systémové zadání požadující české věcné shrnutí silných stránek, slabin, opakovaných motivů a celkové spokojenosti.
- Prompt výslovně zakazuje vyvozovat historické trendy z komentářů jediného semestru.
- Pro každý předmět zavolá `call_gpt`; v offline režimu se API nevolá.

**Výstup:** `text_summaries_df` s identifikací předmětu, výsledným textem `text_summary` a počtem platných textových odpovědí `n_text`.


In [ ]:
# LLM call 1: narrative summary of cleaned anonymous text responses for one course.
def summarize_text_course(batch):
    lines = [f"{i+1}) {txt}" for i, txt in enumerate(batch["responses"])]
    combined = "\n".join(lines)
    n_text = len(batch["responses"])
    system_msg = {
        "role": "system",
        "content": (
            "Jsi analytik studentských anket. Shrň textové odpovědi ke kurzu za uvedený semestr. Piš česky, věcně, 3-5 vět, jeden odstavec. Ve všech větách používej správnou českou diakritiku. "
            "Pokud je odpovědí méně než 5, první věta má říct, že shrnutí je orientační, a uvést přesný počet odpovědí. "
            "Vypíchni silné stránky, slabiny, opakující se motivy a celkovou spokojenost. Pokud jsou protichůdné názory, uveď je. "
            "Nevytvářej trendové závěry z jednoho semestru."
        ),
    }
    user_msg = {
        "role": "user",
        "content": (
            f"Kurz: {batch['kod']} - {batch['nazev']}\n"
            f"Období: {batch['obdobi']}\n"
            f"Počet textových odpovědí: {n_text}.\n"
            f"Textové odpovědi studentů (anonymně):\n{combined}\n\n"
            "Napiš shrnutí podle zadání."
        ),
    }
    return call_gpt([system_msg, user_msg])


# Run text summarization only for courses that passed the report response threshold.
text_summaries = []
for batch in text_batches:
    summary = summarize_text_course(batch)
    text_summaries.append(
        {
            "kod_predmetu_studenta": batch["kod"],
            "nazev_predmetu_studenta": batch["nazev"],
            "text_summary": summary,
            "n_text": len(batch["responses"]),
        }
    )
text_summaries_df = pd.DataFrame(text_summaries, columns=[
    'kod_predmetu_studenta', 'nazev_predmetu_studenta', 'text_summary', 'n_text'
])
text_summaries_df.head()


### Numeric aggregation, trend context + GPT commentary
Numeric commentary uses report-semester aggregates plus local historical trend/anomaly context.


#### Co dělá následující buňka

**Vstupy:** Číselné agregace za reportovaný semestr, lokální trendové výsledky, anomálie a agregace podle vyučujících.

**Postup:**
- `summarize_numeric_course` připraví souhrn statistik jednotlivých otázek a přidá lokálně vypočtený trendový a rizikový kontext. Model z těchto podkladů vytvoří český komentář k číselnému hodnocení.
- `teacher_report_visibility_mask` určuje, kteří vyučující mají být v reportu viditelní podle skóre nebo výrazného poklesu.
- `summarize_teacher_comparison_course` vytváří srovnání viditelných vyučujících a upozorňuje na omezení spolehlivosti.
- GPT nedopočítává trendy; dostává již připravené lokální statistiky a má je pouze věcně interpretovat.

**Výstupy:** `numeric_commentaries_df` a `teacher_commentaries_df`, které obsahují textové komentáře a související počty odpovědí.


In [ ]:
# LLM call 2: narrative explanation of local numeric aggregates, trends and risk flags.
def summarize_numeric_course(course_df: pd.DataFrame, course_label: str, total_answers: int, trend_row: dict | None, anomaly_row: dict | None):
    stats_records = []
    for row in course_df.itertuples():
        stats_records.append(
            {
                "otazka": row.otazka,
                "otazka_plny_text": row.otazka_plny_text,
                "count": int(row.count),
                "mean": round(row.mean, 2) if not pd.isna(row.mean) else None,
                "median": round(row.median, 2) if not pd.isna(row.median) else None,
                "std": round(row.std, 2) if not pd.isna(row.std) else None,
                "min": row.min,
                "max": row.max,
            }
        )
    stats_json = json.dumps(stats_records, ensure_ascii=False)
    trend_json = json.dumps(trend_row or {}, ensure_ascii=False)
    anomaly_json = json.dumps(anomaly_row or {}, ensure_ascii=False)

    system_msg = {
        "role": "system",
        "content": (
            "Jsi analytik studentských anket. Vylož agregovaná číselná hodnocení kurzu pro reportovaný semestr. Piš česky, stručně 3-5 vět, jeden odstavec. Ve všech větách používej správnou českou diakritiku. "
            "Pokud je méně než 50 odpovědí, první věta má upozornit, že shrnutí je orientační, a uvést počet odpovědí. "
            "Zmiň otázky s nejvyšším a nejnižším průměrem, vysokou variabilitu a jednoduché indikátory rizik. "
            "Trend popisuj pouze tehdy, když historický kontext obsahuje více semestrů; jinak řekni, že trend nelze spolehlivě hodnotit. "
            "Rozlišuj problém kvality výuky od nízké spolehlivosti dat."
        ),
    }
    user_msg = {
        "role": "user",
        "content": (
            f"Kurz: {course_label}. Období: {selected_report_obdobi}. Počet číselných odpovědí: {total_answers}.\n"
            f"Agregovaná čísla po otázkách (JSON):\n{stats_json}\n\n"
            f"Historický kontext kurzu (JSON):\n{trend_json}\n\n"
            f"Indikátory rizik a spolehlivosti dat (JSON):\n{anomaly_json}\n\n"
            "Napiš komentář podle zadání a použij údaje count/mean/median/std/min/max."
        ),
    }
    return call_gpt([system_msg, user_msg])


numeric_commentaries = []
for (kod, nazev), group in numeric_stats.groupby(["kod_predmetu_studenta", "nazev_predmetu_studenta"]):
    total_answers = int(group['count'].sum())
    trend_row = trend_summary_df[
        (trend_summary_df['kod_predmetu_studenta'] == kod)
        & (trend_summary_df['nazev_predmetu_studenta'] == nazev)
    ]
    anomaly_row = anomaly_flags_df[
        (anomaly_flags_df['kod_predmetu_studenta'] == kod)
        & (anomaly_flags_df['nazev_predmetu_studenta'] == nazev)
    ]
    commentary = summarize_numeric_course(
        group,
        f"{kod} - {nazev}",
        total_answers,
        trend_row.iloc[0].to_dict() if not trend_row.empty else None,
        anomaly_row.iloc[0].to_dict() if not anomaly_row.empty else None,
    )
    numeric_commentaries.append(
        {
            "kod_predmetu_studenta": kod,
            "nazev_predmetu_studenta": nazev,
            "numeric_commentary": commentary,
            "n_numeric": total_answers,
        }
    )
numeric_commentaries_df = pd.DataFrame(numeric_commentaries, columns=[
    'kod_predmetu_studenta', 'nazev_predmetu_studenta', 'numeric_commentary', 'n_numeric'
])
numeric_commentaries_df.head()


# LLM call 3: narrative teacher comparison based on local teacher-level aggregates.
def summarize_teacher_comparison_course(teacher_df: pd.DataFrame, course_label: str, teacher_trend_group: pd.DataFrame):
    records = []
    teacher_df = teacher_df.sort_values(['mean', 'count'], ascending=[False, False])
    for row in teacher_df.itertuples():
        records.append(
            {
                'teacher_full_name': row.teacher_full_name,
                'teacher_department': row.teacher_department,
                'count': int(row.count),
                'mean': round(row.mean, 2) if not pd.isna(row.mean) else None,
                'median': round(row.median, 2) if not pd.isna(row.median) else None,
                'std': round(row.std, 2) if not pd.isna(row.std) else None,
                'min': row.min,
                'max': row.max,
            }
        )

    stats_json = json.dumps(records, ensure_ascii=False)
    teacher_trend_records = teacher_trend_group.to_dict(orient='records') if teacher_trend_group is not None and not teacher_trend_group.empty else []
    teacher_trend_json = json.dumps(teacher_trend_records, ensure_ascii=False)
    total_answers = int(teacher_df['count'].sum())
    n_teachers = int(teacher_df['teacher_full_name'].nunique())

    system_msg = {
        'role': 'system',
        'content': (
            'Jsi analytik studentských anket. Porovnej vyučující v rámci kurzu podle agregovaných číselných hodnocení za reportovaný semestr. Ve všech větách používej správnou českou diakritiku. '
            'Piš česky, věcně, 3-6 vět, jeden odstavec. '
            'Pokud je méně než 20 odpovědí, první věta má upozornit, že shrnutí je orientační, a uvést počet odpovědí. '
            'Pokud je jen 1 vyučující, řekni, že není co porovnat. Nevyvozuj trendy bez historických dat.'
        ),
    }
    user_msg = {
        'role': 'user',
        'content': (
            f"Kurz: {course_label}. Období: {selected_report_obdobi}. Počet číselných odpovědí (souhrn přes vyučující): {total_answers}. Počet vyučujících: {n_teachers}.\n"
            f"Agregovaná čísla po vyučujících za reportovaný semestr (JSON):\n{stats_json}\n\n"
            f"Historický trend vyučujících (JSON):\n{teacher_trend_json}\n\n"
            'Napiš srovnání podle zadání. Nehádej škálu; popisuj relativní rozdíly v průměrech a spolehlivost trendu.'
        ),
    }
    return call_gpt([system_msg, user_msg])



def teacher_report_visibility_mask(teacher_trend_group: pd.DataFrame) -> pd.Series:
    """Show teachers only when score is <=3.0 or when a large semester drop is present."""
    if teacher_trend_group.empty:
        return pd.Series(dtype=bool)
    current_score = pd.to_numeric(teacher_trend_group['teacher_current_mean'], errors='coerce')
    delta_previous = pd.to_numeric(teacher_trend_group['teacher_delta_vs_previous'], errors='coerce')
    return (current_score <= LOW_SCORE_THRESHOLD) | (delta_previous <= SUDDEN_DROP_THRESHOLD)

# Only teachers with low score or a notable drop are expanded in the report text.
teacher_commentaries = []
for (kod, nazev), group in teacher_numeric_stats.groupby(["kod_predmetu_studenta", "nazev_predmetu_studenta"]):
    teacher_trend_group = teacher_trend_df[
        (teacher_trend_df['kod_predmetu_studenta'] == kod)
        & (teacher_trend_df['nazev_predmetu_studenta'] == nazev)
    ].copy()
    visible_teacher_trend_group = teacher_trend_group[teacher_report_visibility_mask(teacher_trend_group)].copy()
    visible_teachers = set(visible_teacher_trend_group['teacher_full_name']) if not visible_teacher_trend_group.empty else set()
    visible_group = group[group['teacher_full_name'].isin(visible_teachers)].copy()

    if visible_group.empty:
        commentary = (
            f"Do reportu nejsou uvedeni vyucujici s aktualnim hodnocenim nad {LOW_SCORE_THRESHOLD:.1f}, "
            f"pokud u nich neni mezisemestralni pokles alespon {abs(SUDDEN_DROP_THRESHOLD):.1f} bodu."
        )
    else:
        commentary = summarize_teacher_comparison_course(visible_group, f"{kod} - {nazev}", visible_teacher_trend_group)
    teacher_commentaries.append(
        {
            'kod_predmetu_studenta': kod,
            'nazev_predmetu_studenta': nazev,
            'teacher_comparison': commentary,
            'n_teacher_numeric': int(visible_group['count'].sum()) if not visible_group.empty else 0,
            'n_teachers': int(visible_group['teacher_full_name'].nunique()) if not visible_group.empty else 0,
        }
    )

teacher_commentaries_df = pd.DataFrame(teacher_commentaries, columns=[
    'kod_predmetu_studenta', 'nazev_predmetu_studenta', 'teacher_comparison', 'n_teacher_numeric', 'n_teachers'
])
teacher_commentaries_df.head()


### Merge summaries and build final report
Top section: manager summary combining text + numeric. Below: per-course blocks with both summaries.

#### Co dělá následující buňka

**Účel:** Sestaví konečný report a všechny výstupní formáty z výsledků předchozích buněk.

**Sestavení reportu:**
- Spojí textová shrnutí, číselné komentáře, srovnání vyučujících, trendy a anomálie do `report_merged`.
- Aplikuje minimální počet odpovědí a pravidla viditelnosti vyučujících.
- Převádí technické příznaky jako `sudden_drop_vs_previous_semester` na čitelné české věty.
- Vytváří textový trend předmětu, detailní trendy vyučujících, souvislý text rizik a kompletní `full_report_text`.

**Souborové výstupy:**
- Pro každý zahrnutý FIS předmět vytvoří samostatný Markdown a PDF report v adresáři `generated_reports/fis/<obdobi>`.
- Připraví Power BI tabulky včetně dimenzí předmětů a období, souhrnů předmětů, vyučujících, rizik, motivů a varování spolehlivosti.
- Při zapnutém preview uloží CSV a XLSX soubory; při `EXPORT_TO_SQL=True` stejné agregované tabulky odešle do SQL.

**Důležitá vlastnost:** CSV/XLSX export zachovává technické sloupce pro filtrování i všechny čitelné textové odstavce použité v PDF reportu.


In [ ]:
# Merge all local statistics and LLM-generated narratives into one report table per course.
merged = (
    text_summaries_df
    .merge(numeric_commentaries_df, on=['kod_predmetu_studenta', 'nazev_predmetu_studenta'], how='outer')
    .merge(teacher_commentaries_df, on=['kod_predmetu_studenta', 'nazev_predmetu_studenta'], how='outer')
    .merge(trend_summary_df, on=['kod_predmetu_studenta', 'nazev_predmetu_studenta'], how='outer')
    .merge(anomaly_flags_df, on=['kod_predmetu_studenta', 'nazev_predmetu_studenta'], how='outer', suffixes=('', '_anomaly'))
    .fillna({
        'text_summary': '',
        'numeric_commentary': '',
        'teacher_comparison': '',
        'potential_risks': '',
        'data_reliability_warnings': '',
        'anomaly_flags': '',
        'trend_note': '',
        'negative_motif_summary': '',
        'n_text': 0,
        'n_numeric': 0,
        'n_teacher_numeric': 0,
        'n_teachers': 0,
        'current_count': 0,
        'previous_count': 0,
        'historical_count': 0,
        'historical_semesters_available': 0,
        'valid_text_count': 0,
        'numeric_answer_count': 0,
        'trend_direction': 'insufficient_history',
        'trend_reliability': 'LOW',
        'trend_risk_flags': '',
        'trend_reliability_warnings': '',
        'trend_interpretation': '',
        'previous_mean': np.nan,
        'historical_average': np.nan,
        'delta_vs_previous': np.nan,
        'delta_vs_historical_average': np.nan,
    })
)
merged = merged.sort_values('nazev_predmetu_studenta')

selected_course_name = TARGET_FACULTY_LABEL
selected_course_label = f"Fakulta {TARGET_FACULTY_LABEL}"
selected_course_program_codes = ", ".join(sorted(report_prepared["kratka_programu_studenta"].dropna().astype(str).unique())) or "n/a"
selected_course_program_names = "; ".join(sorted(report_prepared["nazev_programu_studenta"].dropna().astype(str).unique())) or "n/a"


# v5 reporting rule: the selected course is included only when it has at least REPORT_MIN_RESPONSE_COUNT numeric answers
# in the report semester. Low-n course reports are kept out of Markdown/PDF and Power BI/SQL aggregate export.
merged['report_response_count'] = pd.to_numeric(
    merged['numeric_answer_count'].where(merged['numeric_answer_count'].notna(), merged['n_numeric']),
    errors='coerce',
).fillna(0).astype(int)
report_merged = merged[merged['report_response_count'] >= REPORT_MIN_RESPONSE_COUNT].copy()
excluded_low_n_courses = merged[merged['report_response_count'] < REPORT_MIN_RESPONSE_COUNT].copy()

# The same course threshold is reused for reports, Power BI preview and optional SQL export.
report_course_keys = report_merged[['kod_predmetu_studenta', 'nazev_predmetu_studenta']].drop_duplicates()

def filter_to_report_courses(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty or report_course_keys.empty:
        return df.iloc[0:0].copy()
    return df.merge(report_course_keys, on=['kod_predmetu_studenta', 'nazev_predmetu_studenta'], how='inner')

num_report_for_report = filter_to_report_courses(num_report_prepared)
trend_summary_for_report = filter_to_report_courses(trend_summary_df)
anomaly_flags_for_report = filter_to_report_courses(anomaly_flags_df)
teacher_trend_for_report = filter_to_report_courses(teacher_trend_df)

def filter_teacher_rows_for_report(teacher_df: pd.DataFrame) -> pd.DataFrame:
    if teacher_df.empty:
        return teacher_df.copy()
    current_score = pd.to_numeric(teacher_df['teacher_current_mean'], errors='coerce')
    delta_previous = pd.to_numeric(teacher_df['teacher_delta_vs_previous'], errors='coerce')
    visible = (current_score <= LOW_SCORE_THRESHOLD) | (delta_previous <= SUDDEN_DROP_THRESHOLD)
    return teacher_df[visible].copy()

teacher_trend_visible_for_report = filter_teacher_rows_for_report(teacher_trend_for_report)
negative_motif_stats_for_report = filter_to_report_courses(negative_motif_stats)

print(f"Selected course included in reports (n>={REPORT_MIN_RESPONSE_COUNT}): {len(report_merged):,}")
print(f"Teacher rows included in reports (score<={LOW_SCORE_THRESHOLD:.1f} or delta<={SUDDEN_DROP_THRESHOLD:.1f}): {len(teacher_trend_visible_for_report):,}")
print(f"Selected course excluded from reports because n<{REPORT_MIN_RESPONSE_COUNT}: {len(excluded_low_n_courses):,}")

course_numeric = (
    num_report_for_report
    .groupby(['otazka', 'otazka_plny_text'])[NUMERIC_COLUMN]
    .agg(['count', 'mean', 'median', 'std', 'min', 'max'])
    .reset_index()
)
course_numeric_records = course_numeric.to_dict(orient='records')
course_numeric_json = json.dumps(course_numeric_records, ensure_ascii=False)
course_trend_records = trend_summary_for_report.to_dict(orient='records')
course_trend_json = json.dumps(course_trend_records, ensure_ascii=False)
course_anomaly_records = anomaly_flags_for_report.to_dict(orient='records')
course_anomaly_json = json.dumps(course_anomaly_records, ensure_ascii=False)


def fmt_num(value, digits=2):
    return 'n/a' if pd.isna(value) else f"{float(value):.{digits}f}"


def fmt_delta(value, digits=2):
    return 'n/a' if pd.isna(value) else f"{float(value):+.{digits}f}"


def describe_change(value, subject='Aktuální výsledek') -> str:
    if pd.isna(value):
        return f'{subject} nelze porovnat, protože chybí potřebná data.'
    value = float(value)
    if abs(value) < 0.01:
        return f'{subject} se prakticky nezměnil.'
    direction = 'vyšší' if value > 0 else 'nižší'
    return f'{subject} je o {abs(value):.2f} bodu {direction}.'


def trend_direction_sentence(direction: str) -> str:
    return {
        'improving': 'Celkový vývoj hodnocení naznačuje zlepšení.',
        'deteriorating': 'Celkový vývoj hodnocení naznačuje zhoršení.',
        'volatile': 'Hodnocení mezi semestry výrazně kolísá.',
        'stable': 'Hodnocení je mezi semestry přibližně stabilní.',
        'insufficient_history': 'Pro posouzení vývoje není dostatek starších dat.',
        'insufficient_sample_size': 'Pro spolehlivé posouzení vývoje není dostatek odpovědí.',
    }.get(str(direction), 'Vývoj hodnocení nelze jednoznačně posoudit.')


def reliability_sentence(reliability: str) -> str:
    return {
        'HIGH': 'Spolehlivost tohoto porovnání je vysoká.',
        'MEDIUM': 'Spolehlivost tohoto porovnání je střední, proto je vhodné výsledek interpretovat opatrně.',
        'LOW': 'Spolehlivost tohoto porovnání je nízká a závěr je pouze orientační.',
    }.get(str(reliability), 'Spolehlivost tohoto porovnání nelze určit.')


def format_course_trend(row) -> str:
    sentences = [f"V aktuálním období je průměrné hodnocení {fmt_num(row.current_mean)} z {int(row.current_count)} odpovědí."]
    if row.previous_obdobi and pd.notna(row.previous_mean):
        sentences.append(
            f"V předchozím semestru {row.previous_obdobi} bylo průměrné hodnocení {fmt_num(row.previous_mean)} z {int(row.previous_count)} odpovědí. "
            + describe_change(row.delta_vs_previous)
        )
    if pd.notna(row.historical_average):
        sentences.append(
            f"Průměr starších období je {fmt_num(row.historical_average)} na základě {int(row.historical_count)} odpovědí ze {int(row.historical_semesters_available)} semestrů. "
            + describe_change(row.delta_vs_historical_average)
        )
    if pd.notna(row.rolling_3_semester_average):
        sentences.append(
            f"Průměr posledních tří předchozích semestrů je {fmt_num(row.rolling_3_semester_average)}. "
            + describe_change(row.delta_vs_rolling_3_semester_average)
        )
    sentences.append(
        f"Nejlepší dostupné hodnocení bylo v semestru {row.best_historical_semester or 'neznámém'} ({fmt_num(row.best_historical_mean)}), "
        f"nejslabší v semestru {row.worst_historical_semester or 'neznámém'} ({fmt_num(row.worst_historical_mean)})."
    )
    sentences.append(trend_direction_sentence(row.trend_direction) + ' ' + reliability_sentence(row.trend_reliability))
    return " ".join(sentences)


RISK_LABELS = {
    'assessment': 'způsob hodnocení a testy',
    'clarity': 'srozumitelnost výuky',
    'communication': 'komunikace',
    'materials': 'studijní materiály',
    'organization': 'organizace předmětu',
    'workload': 'studijní zátěž',
}


def format_risk_flag(flag: str) -> str:
    flag = str(flag).strip()
    teacher = ''
    if ': ' in flag:
        teacher, flag = flag.split(': ', 1)
    subject = f'U vyučujícího {teacher}' if teacher else 'U předmětu'
    value_match = re.search(r'(?:delta|mean|std|gap|n)=(-?\d+(?:\.\d+)?)', flag)
    value = float(value_match.group(1)) if value_match else np.nan

    if flag.startswith(('sudden_drop_vs_previous_semester', 'teacher_score_drop')):
        return f'{subject} kleslo hodnocení oproti předchozímu semestru o {abs(value):.2f} bodu.'
    if flag.startswith('below_historical_baseline'):
        return f'{subject} je aktuální hodnocení o {abs(value):.2f} bodu nižší než dlouhodobý průměr.'
    if flag.startswith('persistent_deterioration'):
        return f'{subject} se hodnocení zhoršuje opakovaně ve více semestrech.'
    if flag.startswith('volatile_scores'):
        return f'{subject} hodnocení mezi semestry výrazně kolísá.'
    if flag.startswith('improving_after_previous_drop'):
        return f'{subject} se hodnocení po předchozím poklesu zlepšuje.'
    if flag.startswith('teacher_below_course_average'):
        return f'{subject} je hodnocení o {abs(value):.2f} bodu nižší než průměr předmětu.'
    if flag.startswith('low_average_score'):
        return f'Průměrné hodnocení předmětu je nízké ({value:.2f}).'
    if flag.startswith('high_polarization'):
        return f'Odpovědi studentů se výrazně rozcházejí; jejich zkušenosti nejsou jednotné.'
    if flag.startswith('large_teacher_difference'):
        return f'Mezi hodnocením jednotlivých vyučujících je výrazný rozdíl {abs(value):.2f} bodu.'
    if flag.startswith('repeated_negative_motifs'):
        motifs = []
        for motif, count in re.findall(r'(\w+):(\d+)', flag):
            motifs.append(f"{RISK_LABELS.get(motif, motif)} ({count} zmínek)")
        return 'V textových odpovědích se opakovaně objevují výhrady k těmto oblastem: ' + ', '.join(motifs) + '.'
    return 'Bylo zjištěno další upozornění, které vyžaduje podrobnější prověření.'


def format_risks(flags: str) -> str:
    if not str(flags).strip():
        return 'Nebyla zjištěna žádná výrazná rizika.'
    sentences = []
    for flag in str(flags).split(';'):
        sentence = format_risk_flag(flag)
        if sentence not in sentences:
            sentences.append(sentence)
    return ' '.join(sentences)


def format_reliability_warnings(flags: str) -> str:
    if not str(flags).strip():
        return 'Data jsou bez zvláštního varování.'
    messages = []
    for flag in str(flags).split(';'):
        flag = flag.strip()
        if flag.startswith('low_sample_size'):
            messages.append('Počet odpovědí je nízký, proto je nutné výsledky interpretovat opatrně.')
        elif flag.startswith('suspiciously_perfect_low_n'):
            messages.append('Malé množství odpovědí je neobvykle jednotné, proto nemusí být reprezentativní.')
        elif flag == 'missing_teacher_name':
            messages.append('U části odpovědí chybí jméno vyučujícího.')
        elif flag == 'missing_text_feedback':
            messages.append('Textových odpovědí je málo nebo zcela chybí.')
        elif flag == 'low_trend_reliability':
            messages.append('Dostupná historie neumožňuje spolehlivě posoudit vývoj hodnocení.')
        else:
            messages.append('Data obsahují další omezení, které vyžaduje podrobnější prověření.')
    return ' '.join(dict.fromkeys(messages))


def format_teacher_trend_lines(kod: str, nazev: str) -> list[str]:
    group = teacher_trend_visible_for_report[
        (teacher_trend_visible_for_report['kod_predmetu_studenta'] == kod)
        & (teacher_trend_visible_for_report['nazev_predmetu_studenta'] == nazev)
    ].copy()
    if group.empty:
        return ['- Trend vyučujících nelze vyhodnotit: v reportovaném semestru nejsou dostupná data za jednotlivé vyučující.']
    lines = []
    for row in group.sort_values(['teacher_full_name']).itertuples():
        teacher_label = row.teacher_full_name
        if teacher_label == 'missing_teacher_name':
            teacher_label = 'Neznámý vyučující (jméno chybí ve zdrojových datech)'
        line = f"- {teacher_label}: V aktuálním období je průměrné hodnocení {fmt_num(row.teacher_current_mean)} z {int(row.teacher_current_count)} odpovědí."
        if row.teacher_previous_obdobi and pd.notna(row.teacher_previous_mean):
            line += (
                f" V předchozím semestru {row.teacher_previous_obdobi} bylo průměrné hodnocení {fmt_num(row.teacher_previous_mean)}. "
                + describe_change(row.teacher_delta_vs_previous, 'Aktuální hodnocení')
            )
        if pd.notna(row.teacher_historical_average):
            line += (
                f" Dlouhodobý průměr je {fmt_num(row.teacher_historical_average)}. "
                + describe_change(row.teacher_delta_vs_historical_average, 'Aktuální hodnocení')
            )
        line += f" {trend_direction_sentence(row.teacher_trend_direction)} {reliability_sentence(row.teacher_trend_reliability)}"
        if row.teacher_trend_flags:
            teacher_flags = '; '.join(f'{teacher_label}: {flag.strip()}' for flag in str(row.teacher_trend_flags).split(';') if flag.strip())
            line += f" Upozornění: {format_risks(teacher_flags)}"
        if row.teacher_reliability_warnings:
            line += f" Omezení dat: {format_reliability_warnings(row.teacher_reliability_warnings)}"
        lines.append(line)
    return lines


def build_course_trend_summary(trend_df: pd.DataFrame, motif_df: pd.DataFrame) -> str:
    if trend_df.empty:
        return 'Trendove shrnuti predmetu nelze vytvorit: nejsou dostupna historicka obdobi pro vybrany predmet.'
    evaluable = trend_df[trend_df['delta_vs_previous'].notna()].copy()
    lines = []
    if not evaluable.empty:
        evaluable = evaluable[
            (evaluable['current_count'] >= MIN_COUNT_FOR_ANOMALY)
            & (evaluable['delta_vs_previous'].abs() >= abs(SUDDEN_DROP_THRESHOLD))
        ]
        improved = evaluable[evaluable['delta_vs_previous'] > 0].sort_values('delta_vs_previous', ascending=False).head(3)
        deteriorated = evaluable[evaluable['delta_vs_previous'] < 0].sort_values('delta_vs_previous', ascending=True).head(3)
        if improved.empty:
            lines.append('Nejvetsi zlepseni proti minulemu semestru: nezjisteno.')
        else:
            lines.append('Nejvetsi zlepseni proti minulemu semestru: ' + '; '.join(
                f"{r.kod_predmetu_studenta} - {r.nazev_predmetu_studenta} ({fmt_delta(r.delta_vs_previous)}, spolehlivost {r.trend_reliability})"
                for r in improved.itertuples()
            ))
        if deteriorated.empty:
            lines.append('Nejvetsi pokles proti minulemu semestru: nezjisten.')
        else:
            lines.append('Nejvetsi pokles proti minulemu semestru: ' + '; '.join(
                f"{r.kod_predmetu_studenta} - {r.nazev_predmetu_studenta} ({fmt_delta(r.delta_vs_previous)}, spolehlivost {r.trend_reliability})"
                for r in deteriorated.itertuples()
            ))
    stable = trend_df[trend_df['trend_direction'].eq('stable')]
    insufficient = trend_df[trend_df['trend_direction'].isin(['insufficient_history', 'insufficient_sample_size'])]
    lines.append(f"Stabilni bez vyrazne zmeny podle prahu: {len(stable)} zaznam predmetu.")
    lines.append(f"Trend nelze hodnotit robustne kvuli historii nebo vzorku: {len(insufficient)} zaznam predmetu.")
    if motif_df.empty:
        lines.append('Opakovane negativni textove motivy napric semestry: nezjisteny v agregovane motif vrstve.')
    else:
        motif_totals = motif_df.groupby('motif')['count'].sum().sort_values(ascending=False)
        lines.append('Opakovane negativni textove motivy napric semestry: ' + ', '.join(f"{k}:{int(v)}" for k, v in motif_totals.items()))
    return "\n".join(f"- {line}" for line in lines if line.strip())


course_trend_summary = build_course_trend_summary(trend_summary_for_report, negative_motif_stats_for_report)


def manager_summary_text(merged_df: pd.DataFrame):
    blocks = []
    for row in merged_df.itertuples():
        blocks.append(
            f"Predmet: {row.kod_predmetu_studenta} - {row.nazev_predmetu_studenta}\n"
            f"Textove shrnuti (odpovedi {int(row.n_text)}): {row.text_summary}\n"
            f"Motivy bez surovych komentaru: {row.negative_motif_summary or 'nezjisteny'}"
        )
    combined = "\n\n".join(blocks)
    system_msg = {
        'role': 'system',
        'content': (
            'Jsi manazersky analytik studentskych anket. Vytvor shrnuti pouze z textovych odpovedi za reportovany semestr. '
            'Formatuj presne takto:\nPozitiva: ...\nNegativa: ...\nDoporuceni: ... '
            'Kazda sekce 2-3 vety. Komunikuj nejistotu pri malem poctu textu a nevyvozuj trend, pokud neni dolozen. '
            'U pozitiv a negativ uved konkretni predmety a vyucujici, pokud jsou z textu zrejmi.'
        ),
    }
    user_msg = {
        'role': 'user',
        'content': (
            f"Predmet: {selected_course_label}\n"
            f"Programy studentu v reportovanem obdobi: {selected_course_program_codes}\n"
            f"Reportovane obdobi: {selected_report_obdobi}\n"
            f"Historicke okno: {selected_obdobi or 'vsechna dostupna obdobi'}\n\n"
            f"Shrnuti vybraneho predmetu (text) vcetne agregovanych motivu:\n{combined}\n"
        ),
    }
    return call_gpt([system_msg, user_msg])


def manager_summary_numeric(merged_df: pd.DataFrame):
    num_blocks = []
    for row in merged_df.itertuples():
        num_blocks.append(
            f"Predmet: {row.kod_predmetu_studenta} - {row.nazev_predmetu_studenta} (odpovedi {int(row.n_numeric)})\n"
            f"Prehled z ciselneho hodnoceni: {row.numeric_commentary}\n"
            f"Trend: {row.trend_note}\n"
            f"Rizika: {row.potential_risks or 'bez oznacenych rizik'}\n"
            f"Spolehlivost dat: {row.data_reliability_warnings or 'bez zvlastniho varovani'}"
        )
    combined = "\n\n".join(num_blocks)
    system_msg = {
        'role': 'system',
        'content': (
            'Jsi manazersky analytik studentskych anket. Vytvor shrnuti z ciselnych odpovedi a lokalne vypoctenych trendu. '
            'Formatuj presne takto:\nPozitiva: ...\nNegativa: ...\nDoporuceni: ... '
            'Kazda sekce 2-3 vety. Jasne rozlisuj signaly kvality vyuky od nizke spolehlivosti dat. '
            'Nepouzivej formulace typu stabilni trend, pokud je dostupny jen jeden semestr. Pouzivej opatrny jazyk.'
        ),
    }
    user_msg = {
        'role': 'user',
        'content': (
            f"Predmet: {selected_course_label}\n"
            f"Programy studentu v reportovanem obdobi: {selected_course_program_codes}\n"
            f"Reportovane obdobi: {selected_report_obdobi}\n"
            f"Historicke okno: {selected_obdobi or 'vsechna dostupna obdobi'}\n"
            f"Cisla vybraneho predmetu za reportovane obdobi (JSON): {course_numeric_json}\n\n"
            f"Trendove kontexty (JSON): {course_trend_json}\n\n"
            f"Teacher-level trendy (JSON): {json.dumps(teacher_trend_visible_for_report.to_dict(orient='records'), ensure_ascii=False)}\n\n"
            f"Indikatory rizik a spolehlivosti (JSON): {course_anomaly_json}\n\n"
            f"Deterministicke trendove shrnuti predmetu:\n{course_trend_summary}\n\n"
            f"Shrnuti vybraneho predmetu (cisla):\n{combined}\n"
        ),
    }
    return call_gpt([system_msg, user_msg])


def slugify(value: str) -> str:
    return re.sub(r'[^A-Za-z0-9_-]+', '_', value).strip('_').lower()


def build_markdown_report(course_df: pd.DataFrame, course_label: str) -> str:
    report_lines = [
        f"# Předmět: {course_label}",
        f"\nObdobí: {selected_report_obdobi}; historické okno: {selected_obdobi or 'všechna dostupná období'}",
        "\nPoznámka: Textové části reportu byly vygenerovány pomocí Azure OpenAI na základě lokálně vypočtených agregací a vyčištěných anketních odpovědí; slouží jako podpůrný analytický výstup.",
        "\n## Vybraný předmět",
    ]
    for row in course_df.itertuples():
        report_lines.append(f"\n### {row.kod_predmetu_studenta} - {row.nazev_predmetu_studenta} (textové odpovědi {int(row.n_text)}, číselné odpovědi {int(row.n_numeric)}, vyučující {int(row.n_teachers)})")
        report_lines.append(f"- Přehled z číselného hodnocení: {row.numeric_commentary}")
        report_lines.append("\n#### Trendový kontext")
        report_lines.append(format_course_trend(row))
        report_lines.append(f"- Textové shrnutí: {row.text_summary}")
        report_lines.append(f"- Srovnání vyučujících (číselné odpovědi): {row.teacher_comparison}")
        report_lines.append("\n#### Srovnání vyučujících a trend")
        report_lines.extend(format_teacher_trend_lines(row.kod_predmetu_studenta, row.nazev_predmetu_studenta))
        report_lines.append(f"- Možná rizika: {format_risks(row.potential_risks)}")
    return "\n".join(report_lines)


def build_pdf(output_path: str, title: str, subtitle: str, merged_df: pd.DataFrame):
    try:
        from reportlab.lib.pagesizes import A4
        from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak, Image
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.lib.enums import TA_LEFT
        from reportlab.lib.units import cm
        from reportlab.pdfbase import pdfmetrics
        from reportlab.pdfbase.ttfonts import TTFont
        from xml.sax.saxutils import escape
    except ImportError:
        print('reportlab neni nainstalovany; spust pip install reportlab pro PDF vystup')
        return False

    def get_czech_font():
        candidates = [
            'DejaVuSans.ttf',
            '/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf',
            '/System/Library/Fonts/Supplemental/Arial.ttf',
            '/System/Library/Fonts/Supplemental/Arial Unicode.ttf',
            '/Library/Fonts/Arial Unicode.ttf',
            'C:/Windows/Fonts/arial.ttf',
        ]
        for cand in candidates:
            p = Path(cand)
            if p.exists():
                try:
                    pdfmetrics.registerFont(TTFont('CustomFont', str(p)))
                    return 'CustomFont'
                except Exception:
                    continue
        warnings.warn('No Unicode-capable TTF font found for PDF export; Czech glyphs may be missing.', RuntimeWarning)
        return 'Helvetica'

    font_name = get_czech_font()
    styles = getSampleStyleSheet()
    styles.add(ParagraphStyle(name='TitleCustom', parent=styles['Heading1'], fontName=font_name, fontSize=18, leading=22, spaceAfter=12))
    styles.add(ParagraphStyle(name='SubTitle', parent=styles['Normal'], fontName=font_name, fontSize=11, leading=14, spaceAfter=12))
    styles.add(ParagraphStyle(name='Section', parent=styles['Heading2'], fontName=font_name, fontSize=14, leading=18, spaceAfter=8))
    styles.add(ParagraphStyle(name='Body', parent=styles['Normal'], fontName=font_name, fontSize=10, leading=14, spaceAfter=8, alignment=TA_LEFT))
    styles.add(ParagraphStyle(name='Course', parent=styles['Heading3'], fontName=font_name, fontSize=12, leading=16, spaceAfter=6))

    doc = SimpleDocTemplate(str(output_path), pagesize=A4, rightMargin=2*cm, leftMargin=2*cm, topMargin=2*cm, bottomMargin=2*cm)
    story = []
    logo_path = Path('VSE_logo_black.png')
    if logo_path.exists():
        logo = Image(str(logo_path), width=1.4*cm, height=1*cm, mask='auto')
        logo._restrictSize(1.4*cm, 1*cm)
        logo.hAlign = 'RIGHT'
        story.extend([logo, Spacer(1, 0.4*cm)])

    story.append(Paragraph(escape(title), styles['TitleCustom']))
    story.append(Paragraph(escape(subtitle), styles['SubTitle']))
    story.append(Paragraph(escape('Poznámka: Textové části reportu byly vygenerovány pomocí Azure OpenAI na základě lokálně vypočtených agregací a vyčištěných anketních odpovědí.'), styles['SubTitle']))
    story.append(Spacer(1, 0.2*cm))
    story.append(Paragraph('Vybraný předmět', styles['Section']))
    for idx, row in enumerate(merged_df.itertuples(), start=1):
        heading = f"{row.kod_predmetu_studenta} - {row.nazev_predmetu_studenta} (textové odpovědi {int(row.n_text)}, číselné odpovědi {int(row.n_numeric)}, vyučující {int(row.n_teachers)})"
        story.append(Paragraph(escape(heading), styles['Course']))
        story.append(Paragraph(f"Přehled z číselného hodnocení: {escape(str(row.numeric_commentary))}", styles['Body']))
        story.append(Paragraph(f"<b>Trendový kontext:</b> {escape(format_course_trend(row))}", styles['Body']))
        story.append(Paragraph(f"Textové shrnutí: {escape(str(row.text_summary))}", styles['Body']))
        story.append(Paragraph(f"Srovnání vyučujících (číselné odpovědi): {escape(str(row.teacher_comparison))}", styles['Body']))
        teacher_lines = '<br/>'.join(escape(line) for line in format_teacher_trend_lines(row.kod_predmetu_studenta, row.nazev_predmetu_studenta))
        story.append(Paragraph('Srovnání vyučujících a trend:<br/>' + teacher_lines, styles['Body']))
        story.append(Paragraph(f"Možná rizika: {escape(format_risks(row.potential_risks))}", styles['Body']))
        story.append(Spacer(1, 0.2*cm))
        if idx % 6 == 0:
            story.append(PageBreak())
    doc.build(story)
    return True


# Markdown/PDF generation adds the visible AI disclosure line to every course report.
def generate_course_reports(merged_df: pd.DataFrame, output_dir: Path) -> pd.DataFrame:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    generated = []
    for (course_code, course_name), course_df in merged_df.groupby(['kod_predmetu_studenta', 'nazev_predmetu_studenta'], sort=True):
        course_label = f"{course_code} - {course_name}"
        slug_base = slugify(f"{course_code}_{selected_report_obdobi}_historie")
        md_path = output_dir / f"{slug_base}.md"
        pdf_path = output_dir / f"{slug_base}.pdf"
        md_path.write_text(build_markdown_report(course_df, course_label), encoding='utf-8')
        pdf_created = build_pdf(
            pdf_path,
            title=f"Předmět: {course_label}",
            subtitle=f"Období: {selected_report_obdobi}; historické okno: {selected_obdobi or 'všechna dostupná období'}",
            merged_df=course_df,
        )
        generated.append({
            'course_code': course_code,
            'course_name': course_name,
            'markdown_path': str(md_path),
            'pdf_path': str(pdf_path) if pdf_created else None,
        })
    return pd.DataFrame(generated, columns=['course_code', 'course_name', 'markdown_path', 'pdf_path'])


def get_sql_engine():
    """Create SQLAlchemy engine from environment variables; never hardcode credentials in the notebook."""
    try:
        from sqlalchemy import create_engine
        from sqlalchemy.engine import URL
    except ImportError as exc:
        raise ImportError("SQL export requires sqlalchemy. Install it with `%pip install sqlalchemy pyodbc`.") from exc

    required = ["AZURE_SQL_SERVER", "AZURE_SQL_DATABASE", "AZURE_SQL_USERNAME", "AZURE_SQL_PASSWORD"]
    missing = [name for name in required if not os.getenv(name)]
    if missing:
        raise RuntimeError(f"Missing SQL environment variables: {', '.join(missing)}")

    driver = os.getenv("AZURE_SQL_DRIVER", "ODBC Driver 18 for SQL Server")
    connection_url = URL.create(
        "mssql+pyodbc",
        username=os.getenv("AZURE_SQL_USERNAME"),
        password=os.getenv("AZURE_SQL_PASSWORD"),
        host=os.getenv("AZURE_SQL_SERVER"),
        database=os.getenv("AZURE_SQL_DATABASE"),
        query={"driver": driver, "Encrypt": "yes", "TrustServerCertificate": "no"},
    )
    return create_engine(connection_url, fast_executemany=True)


def _clean_sql_df(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize pandas nulls and timezone-aware datetimes before SQL/Excel export."""
    out = df.copy()
    for col in out.columns:
        if pd.api.types.is_datetime64tz_dtype(out[col]):
            out[col] = out[col].dt.tz_convert("UTC").dt.tz_localize(None)
            continue
        if pd.api.types.is_datetime64_any_dtype(out[col]):
            continue
        if out[col].dtype == object:
            out[col] = out[col].map(
                lambda value: value.tz_convert("UTC").tz_localize(None)
                if isinstance(value, pd.Timestamp) and value.tzinfo is not None
                else value
            )
            out[col] = out[col].where(out[col].notna(), None)
    return out.replace({np.nan: None})


def export_report_tables_to_sql(tables: dict[str, pd.DataFrame], engine=None, table_prefix: str = SQL_TABLE_PREFIX, if_exists: str = SQL_IF_EXISTS):
    """Export Power BI-facing aggregate tables with pandas.to_sql."""
    engine = engine or get_sql_engine()
    exported = {}
    for name, table in tables.items():
        sql_name = f"{table_prefix}{name}"
        _clean_sql_df(table).to_sql(sql_name, engine, if_exists=if_exists, index=False)
        exported[sql_name] = len(table)
    return exported


def export_report_tables_locally(tables: dict[str, pd.DataFrame], output_dir: Path = SQL_PREVIEW_DIR):
    """Save local CSV previews of the aggregate SQL export tables."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    saved = {}
    for name, table in tables.items():
        path = output_dir / f"{SQL_TABLE_PREFIX}{name}.csv"
        _clean_sql_df(table).to_csv(path, index=False, encoding="utf-8-sig")
        saved[str(path)] = len(table)

    try:
        xlsx_path = output_dir / f"{SQL_TABLE_PREFIX}powerbi_export_preview.xlsx"
        with pd.ExcelWriter(xlsx_path) as writer:
            for name, table in tables.items():
                _clean_sql_df(table).to_excel(writer, sheet_name=name[:31], index=False)
        saved[str(xlsx_path)] = sum(len(table) for table in tables.values())
    except ImportError:
        print("Excel preview nebyl vytvoren; pro XLSX nainstaluj openpyxl. CSV soubory jsou ulozene.")
    except ValueError as exc:
        print(f"Excel preview nebyl vytvoren kvuli formatu dat: {exc}. CSV soubory jsou ulozene.")
    return saved


def display_powerbi_table_preview(tables: dict[str, pd.DataFrame], rows: int = 5):
    """Print table shapes and display a small sample from each export table."""
    for name, table in tables.items():
        print(f"\n{SQL_TABLE_PREFIX}{name}: {len(table):,} rows x {len(table.columns):,} columns")
        display(table.head(rows))




# === Multi-course Power BI export (vsechny predmety v jednom behu) ===
import re as _re
from datetime import datetime, timezone


def build_dim_course(catalog: pd.DataFrame) -> pd.DataFrame:
    """Dimenze predmetu z katalogu (1 radek na predmet)."""
    cols = {
        "kod_predmetu_studenta": "course_code",
        "nazev_predmetu_studenta": "course_name",
        "fakulta_predmetu": "faculty",
        "garantujici_pracoviste_predmetu": "department",
    }
    have = [c for c in cols if c in catalog.columns]
    return catalog[have].rename(columns=cols).drop_duplicates("course_code").reset_index(drop=True)


def build_dim_period(periods) -> pd.DataFrame:
    """Dimenze obdobi s ciselnym radicim klicem period_sort (kvuli razeni v Power BI)."""
    rows = []
    for p in sorted({str(x) for x in periods if pd.notna(x)}):
        m = _re.match(r"(ZS|LS)\s+(\d{4})/\d{4}", p)
        if m:
            term, year = m.group(1), int(m.group(2))
            rows.append({"period": p, "semester_term": term, "year_label": m.group(2),
                         "period_sort": year * 10 + SEMESTER_TERM_ORDER.get(term, 0)})
        else:
            rows.append({"period": p, "semester_term": None, "year_label": None, "period_sort": 0})
    return pd.DataFrame(rows)


# Power BI tables contain aggregate/report data only, not raw standalone text responses.
def build_powerbi_export_tables_multi(report_id, generated_at, period):
    """Power BI tabulky pro VSECHNY predmety v report_merged.

    Stejne schema sloupcu jako puvodni build_powerbi_export_tables (append-kompatibilni),
    ale course_id/course_code/course_name se berou z radku, ne z jednoho skalaru.
    """
    course_reports = (report_files
                      .assign(report_id=report_id,
                              course_id=lambda d: d["course_code"],
                              period=period,
                              generated_at=generated_at,
                              historical_window=str(selected_obdobi or "vsechna dostupna obdobi"))
                      [["report_id", "course_id", "course_name", "period", "generated_at",
                        "historical_window", "markdown_path", "pdf_path"]])

    course_summary_source = report_merged.copy()
    course_summary_source["course_heading_text"] = course_summary_source.apply(
        lambda row: (
            f"{row['kod_predmetu_studenta']} - {row['nazev_predmetu_studenta']} "
            f"(textové odpovědi {int(row['n_text'])}, číselné odpovědi {int(row['n_numeric'])}, "
            f"vyučující {int(row['n_teachers'])})"
        ),
        axis=1,
    )
    course_summary_source["course_trend_text"] = course_summary_source.apply(format_course_trend, axis=1)
    course_summary_source["teacher_trend_text"] = course_summary_source.apply(
        lambda row: "\n".join(format_teacher_trend_lines(row["kod_predmetu_studenta"], row["nazev_predmetu_studenta"])),
        axis=1,
    )
    course_summary_source["full_report_text"] = course_summary_source.apply(
        lambda row: build_markdown_report(
            pd.DataFrame([row]),
            f"{row['kod_predmetu_studenta']} - {row['nazev_predmetu_studenta']}",
        ),
        axis=1,
    )

    course_summary = course_summary_source.assign(
        report_id=report_id,
        period=period,
        potential_risks_text=lambda d: d["potential_risks"].map(format_risks),
        data_reliability_warnings_text=lambda d: d["data_reliability_warnings"].map(format_reliability_warnings),
    )[[
        "report_id", "period",
        "kod_predmetu_studenta", "nazev_predmetu_studenta",
        "n_numeric", "n_text", "current_mean", "previous_mean", "historical_average",
        "delta_vs_previous", "delta_vs_historical_average", "trend_direction", "trend_reliability",
        "course_heading_text", "numeric_commentary", "course_trend_text", "text_summary",
        "teacher_comparison", "teacher_trend_text", "potential_risks", "potential_risks_text",
        "data_reliability_warnings", "data_reliability_warnings_text", "negative_motif_summary",
        "full_report_text",
    ]].rename(columns={
        "kod_predmetu_studenta": "course_code",
        "nazev_predmetu_studenta": "course_name",
        "n_numeric": "numeric_response_count",
        "n_text": "text_response_count",
        "current_mean": "current_score",
        "previous_mean": "previous_score",
        "historical_average": "historical_avg",
        "delta_vs_historical_average": "delta_vs_historical",
        "numeric_commentary": "summary_numeric",
        "text_summary": "summary_text",
        "teacher_comparison": "teacher_comparison_text",
    })
    course_summary.insert(1, "course_id", course_summary["course_code"])

    if not teacher_trend_visible_for_report.empty:
        teacher_summary = teacher_trend_visible_for_report.assign(report_id=report_id, period=period)[[
            "report_id", "period",
            "kod_predmetu_studenta", "nazev_predmetu_studenta", "teacher_full_name",
            "teacher_current_mean", "teacher_previous_mean", "teacher_historical_average", "teacher_delta_vs_previous",
            "teacher_trend_direction", "teacher_trend_reliability", "teacher_current_count", "teacher_trend_interpretation",
            "teacher_trend_flags", "teacher_reliability_warnings",
        ]].rename(columns={
            "kod_predmetu_studenta": "course_code",
            "nazev_predmetu_studenta": "course_name",
            "teacher_full_name": "teacher_name",
            "teacher_current_mean": "current_score",
            "teacher_previous_mean": "previous_score",
            "teacher_historical_average": "historical_avg",
            "teacher_delta_vs_previous": "delta_vs_previous",
            "teacher_trend_direction": "trend_direction",
            "teacher_trend_reliability": "trend_reliability",
            "teacher_current_count": "response_count",
            "teacher_trend_interpretation": "interpretation",
        })
        teacher_summary.insert(1, "course_id", teacher_summary["course_code"])
    else:
        teacher_summary = pd.DataFrame(columns=[
            "report_id", "course_id", "period", "course_code", "course_name", "teacher_name", "current_score",
            "previous_score", "historical_avg", "delta_vs_previous", "trend_direction", "trend_reliability",
            "response_count", "interpretation", "teacher_trend_flags", "teacher_reliability_warnings"])

    risk_rows = []
    for row in report_merged.itertuples():
        response_count = int(getattr(row, "numeric_answer_count", 0) or 0)
        for risk in [x.strip() for x in str(row.potential_risks).split(";") if x.strip()]:
            risk_code = risk.split(": ", 1)[-1].split()[0]
            severity = "high" if ("sudden_drop" in risk or "low_average_score" in risk) else "medium"
            risk_rows.append({
                "report_id": report_id, "course_id": row.kod_predmetu_studenta, "period": period,
                "course_code": row.kod_predmetu_studenta, "course_name": row.nazev_predmetu_studenta,
                "teacher_name": None, "risk_type": risk_code, "risk_value": risk,
                "risk_text": format_risk_flag(risk), "severity": severity, "reason": risk,
                "response_count": response_count})
    for row in teacher_summary.itertuples(index=False):
        response_count = int(getattr(row, "response_count", 0) or 0)
        if response_count < MIN_COUNT_FOR_ANOMALY:
            continue
        for risk in [x.strip() for x in str(getattr(row, "teacher_trend_flags", "")).split(";") if x.strip()]:
            risk_rows.append({
                "report_id": report_id, "course_id": row.course_code, "period": period,
                "course_code": row.course_code, "course_name": row.course_name,
                "teacher_name": row.teacher_name, "risk_type": risk.split()[0], "risk_value": risk,
                "risk_text": format_risk_flag(f"{row.teacher_name}: {risk}"),
                "severity": "medium", "reason": risk, "response_count": response_count})
    risk_flags = pd.DataFrame(risk_rows, columns=[
        "report_id", "course_id", "period", "course_code", "course_name", "teacher_name",
        "risk_type", "risk_value", "risk_text", "severity", "reason", "response_count"])

    if not negative_motif_stats_for_report.empty:
        negative_motifs = negative_motif_stats_for_report[
            negative_motif_stats_for_report["obdobi"].eq(period)
        ].assign(report_id=report_id, period=period)[[
            "report_id", "period", "kod_predmetu_studenta", "nazev_predmetu_studenta", "motif", "count"
        ]].rename(columns={
            "kod_predmetu_studenta": "course_code",
            "nazev_predmetu_studenta": "course_name",
            "motif": "motif_type", "count": "motif_count"})
        negative_motifs.insert(1, "course_id", negative_motifs["course_code"])
    else:
        negative_motifs = pd.DataFrame(columns=[
            "report_id", "course_id", "period", "course_code", "course_name", "motif_type", "motif_count"])

    warning_rows = []
    for row in report_merged.itertuples():
        response_count = int(getattr(row, "numeric_answer_count", 0) or 0)
        for warning in [x.strip() for x in str(row.data_reliability_warnings).split(";") if x.strip()]:
            warning_rows.append({
                "report_id": report_id, "course_id": row.kod_predmetu_studenta, "period": period,
                "course_code": row.kod_predmetu_studenta, "course_name": row.nazev_predmetu_studenta,
                "warning_type": warning.split()[0], "warning_detail": warning, "response_count": response_count})
    reliability_warnings = pd.DataFrame(warning_rows, columns=[
        "report_id", "course_id", "period", "course_code", "course_name",
        "warning_type", "warning_detail", "response_count"])

    return {
        "dim_course": build_dim_course(target_faculty_catalog),
        "dim_period": build_dim_period(prepared["obdobi"].unique()),
        "course_reports": course_reports,
        "course_summary": course_summary,
        "teacher_summary": teacher_summary,
        "risk_flags": risk_flags,
        "negative_motifs": negative_motifs,
        "reliability_warnings": reliability_warnings,
    }


# Final outputs are scoped by faculty label and report semester.
report_output_dir = Path('generated_reports') / TARGET_FACULTY_LABEL.lower() / slugify(selected_report_obdobi)
report_files = generate_course_reports(report_merged, report_output_dir)
print(f"Markdown/PDF reporty vytvoreny: {len(report_files):,}; adresar: {report_output_dir}")
display(report_files.head())

report_generated_at = pd.Timestamp(datetime.now(timezone.utc)).tz_convert("UTC").tz_localize(None)
report_id = slugify(f"batch_{TARGET_FACULTY_LABEL}_{selected_report_obdobi}_{report_generated_at.strftime('%Y%m%d%H%M%S')}")

powerbi_tables = build_powerbi_export_tables_multi(report_id, report_generated_at, selected_report_obdobi)

display_powerbi_table_preview(powerbi_tables)

if EXPORT_SQL_PREVIEW_FILES:
    batch_dir = SQL_PREVIEW_DIR / f"batch_{report_generated_at:%Y%m%d}"
    preview_files = export_report_tables_locally(powerbi_tables, output_dir=batch_dir)
    print(f"CSV sada pro Power BI ulozena do: {batch_dir}")

if EXPORT_TO_SQL:
    exported_rows = export_report_tables_to_sql(powerbi_tables)
    print(f"SQL export dokoncen: {exported_rows}")
else:
    print("SQL export vypnuty (EXPORT_TO_SQL=False). Tabulky jsou v promenne powerbi_tables.")
